# Lesson 3 — Modeling: fitting, judging, choosing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djnzx/ida-practice-3d/blob/main/practice/3/3-practice.ipynb)

Intellectual Data Analysis — practical part, lesson 3 of 3.

## Learning objectives

By the end of this session you should be able to:

- use any scikit-learn estimator from the same three methods, and say what
  `fit`, `predict` and `transform` each mean;
- choose a metric that matches the question, and read RMSE in the units of
  the target;
- explain why the test set is touched exactly once, and recognise the
  three common ways people touch it more often;
- put every preprocessing step inside a `Pipeline` and say what leaks if
  you do not;
- read a bias-variance curve and identify underfitting and overfitting
  from the gap between training and validation error;
- build precision, recall and F1 from a confusion matrix by hand, and
  state the majority-class baseline before reading any accuracy;
- explain the margin, the support vectors, `C` and `gamma`, and predict
  what each does to a decision boundary;
- state what a clustering result can and cannot establish in the absence
  of ground truth, and choose between inertia, silhouette and BIC.

## Prerequisites

Lesson 1 and Lesson 2. Specifically: `reshape(-1, 1)` (Practice 1 §3.5),
seeded generation (Practice 1 §4.1), the synthetic datasets built in
Practice 1 §6, one-hot against label encoding (Practice 2 §13), fitting a
scaler on training data only (Practice 2 §14.3), and PCA (Practice 2 §15).

## Practicalities

- Written for **90-120 minutes** (2 to 3 academic hours) of class time.
- Measured end-to-end execution time: **about 21 seconds** on a clean
  kernel (reference machine, CPython 3.9, no GPU); the notebook prints
  its own measured time in the last cell of §17. The kernel grid in §12,
  the grid searches in §6 and §12, and the clustering sweeps in §13 are
  the expensive cells; each carries a comment saying what was reduced to
  keep the notebook inside its budget and what the full version would
  cost.
- Homework: `3-homework.ipynb`, 3-4 hours.

## How this lesson fits

Lessons 1 and 2 produced data. This lesson fits models to it and — the
harder half — decides whether to believe the result. The arc is: fit a
model whose right answer you know (§3), learn to measure error (§4),
learn to hold data back so the measurement means something (§5, §6), then
apply that discipline to regression (§7), trees (§8), KNN (§10), SVMs
(§11, §12) and clustering (§13, §14), and close with one honest
end-to-end run (§15).

## 1. Configuration

In [ ]:
import sys
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

SEED = 20250919
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (7.0, 4.5)
plt.rcParams["figure.dpi"] = 110

NOTEBOOK_START = time.time()

print("python      ", sys.version.split()[0])
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("matplotlib  ", matplotlib.__version__)
print("seaborn     ", sns.__version__)
print("scikit-learn", sklearn.__version__)

## 2. The estimator interface

Every object in scikit-learn implements the same small interface. Learn it
once and the remaining eight models in this lesson are a matter of
choosing which one to instantiate.

| Method | Who has it | What it does |
|---|---|---|
| `fit(X, y)` | every estimator | learns parameters from training data; returns the estimator |
| `predict(X)` | classifiers, regressors, clusterers | produces an output per row |
| `transform(X)` | preprocessors, decompositions | rewrites `X`; does not predict |
| `fit_transform(X)` | preprocessors | `fit` then `transform`, in one call |
| `score(X, y)` | supervised estimators | a default metric: accuracy for classifiers, R² for regressors |
| `predict_proba(X)` | most classifiers | a probability per class, not just a label |

Two conventions hold throughout:

- `X` is 2-D, **one row per sample and one column per feature** — which is
  why Practice 1 §3.5 spent a page on `reshape(-1, 1)`.
- Anything learned from data is stored on the estimator with a **trailing
  underscore**: `coef_`, `cluster_centers_`, `feature_importances_`. No
  trailing underscore means you set it; a trailing underscore means `fit`
  set it.

In [ ]:
from sklearn.linear_model import LinearRegression

demo_x = np.array([1.0, 2.0, 3.0, 4.0, 5.0]).reshape(-1, 1)
demo_y = np.array([2.0, 4.0, 6.0, 8.0, 10.0])

demo_model = LinearRegression()
print("before fit, does coef_ exist?", hasattr(demo_model, "coef_"))

demo_model.fit(demo_x, demo_y)
print("after fit, coef_ =", demo_model.coef_, " intercept_ =", round(demo_model.intercept_, 10))
print("predict([[6]]) =", demo_model.predict([[6.0]]))
print("score (R^2)    =", demo_model.score(demo_x, demo_y))

## 3. Linear regression: recovering a known answer

Rebuild the synthetic regression dataset from Practice 1 §6.1. Its
parameters are known, so the model can be graded against the truth rather
than only against its own predictions.

In [ ]:
regression_rng = np.random.default_rng(SEED)

true_a = 1.0
true_b = 3.0
noise_sigma = 1.0
n_regression = 200

x_regression = regression_rng.uniform(0, 5, n_regression)
noise = regression_rng.normal(0, noise_sigma, n_regression)
y_regression = true_a * x_regression + true_b + noise

# sklearn wants a 2-D X: 200 samples, 1 feature. See Lesson 1 section 3.5.
X_regression = x_regression.reshape(-1, 1)

print("x shape", x_regression.shape, "-> X shape", X_regression.shape)
print("true a =", true_a, " true b =", true_b)

In [ ]:
linear = LinearRegression()
linear.fit(X_regression, y_regression)

estimated_a = linear.coef_[0]
estimated_b = linear.intercept_

print(f"true      a = {true_a:.4f}   b = {true_b:.4f}")
print(f"estimated a = {estimated_a:.4f}   b = {estimated_b:.4f}")
print(f"error       {estimated_a - true_a:+.4f}      {estimated_b - true_b:+.4f}")

Close, and not exact. The model fitted 200 noisy points; the noise has
`sigma = 1`, so the line it recovers is an estimate with its own
uncertainty, exactly as the sample mean was an estimate in Practice 1 §4.6.

More data narrows it. This is the same square-root relationship:

In [ ]:
for n in (20, 200, 2_000, 20_000):
    sweep_rng = np.random.default_rng(SEED)
    x_n = sweep_rng.uniform(0, 5, n)
    y_n = true_a * x_n + true_b + sweep_rng.normal(0, noise_sigma, n)
    fitted = LinearRegression().fit(x_n.reshape(-1, 1), y_n)
    print(f"n = {n:6,}   a_hat = {fitted.coef_[0]:7.4f}   b_hat = {fitted.intercept_:7.4f}"
          f"   |error| = {abs(fitted.coef_[0] - true_a):.4f}")

In [ ]:
x_grid = np.linspace(0, 5, 100)

fig, ax = plt.subplots()
ax.scatter(x_regression, y_regression, s=25, alpha=0.6, label="observations")
ax.plot(x_grid, true_a * x_grid + true_b, color="black", linewidth=2.5,
        label=f"true: y = {true_a:.0f}x + {true_b:.0f}")
ax.plot(x_grid, linear.predict(x_grid.reshape(-1, 1)), color="tab:red",
        linestyle="--", linewidth=2.5,
        label=f"fitted: y = {estimated_a:.3f}x + {estimated_b:.3f}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Linear regression recovers the line the data was scattered around")
ax.legend()
plt.show()

The fitted line does not pass through the points. It is not supposed to.
It estimates the conditional mean of `y` given `x`, and the scatter about
it is the noise that was added on purpose.

## 4. Regression metrics

A single number summarising how wrong a model is. There are several, they
penalise different mistakes, and the choice is part of stating the
problem.

Use a real target so the units mean something: hourly bicycle rentals.

In [ ]:
bikes = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/bike_sharing_hourly.csv"
)
bikes["dteday"] = pd.to_datetime(bikes["dteday"])

print("shape:", bikes.shape)
print("target 'cnt' — rentals per hour:")
print(bikes["cnt"].describe().round(2))

In [ ]:
# LOCAL ALTERNATIVE. Run EITHER this cell OR the one above, not both.
# Above reads over the network and is what Colab needs. This one reads
# the same files from your checkout, for the dev-env container. The
# data is identical; only the address differs.
bikes = pd.read_csv("../datasets/bike_sharing_hourly.csv")
bikes["dteday"] = pd.to_datetime(bikes["dteday"])

print("shape:", bikes.shape)
print("target 'cnt' — rentals per hour:")
print(bikes["cnt"].describe().round(2))

### 4.1 Building the feature matrix

`casual` and `registered` sum exactly to `cnt`, so using them as features
would be **target leakage** — predicting a quantity from its own parts.
Drop them, along with the row index and the raw date.

`hr`, `season`, `weathersit`, `mnth` and `weekday` are integer-coded
categories (Practice 2 §8.1), so they get one-hot encoding, not integers
(Practice 2 §13).

In [ ]:
categorical = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
numeric = ["temp", "atemp", "hum", "windspeed"]

bike_features = pd.get_dummies(bikes[categorical].astype("category"), dtype=int)
bike_features[numeric] = bikes[numeric]

X_bikes = bike_features.values.astype(float)
y_bikes = bikes["cnt"].values.astype(float)

print("features:", X_bikes.shape, " target:", y_bikes.shape)
print("leaked columns excluded:", [c for c in ("casual", "registered") if c not in bike_features])

In [ ]:
from sklearn.model_selection import train_test_split

X_bike_tr, X_bike_te, y_bike_tr, y_bike_te = train_test_split(
    X_bikes, y_bikes, test_size=0.25, random_state=SEED
)

bike_model = LinearRegression().fit(X_bike_tr, y_bike_tr)
y_bike_pred = bike_model.predict(X_bike_te)

print("train", X_bike_tr.shape, " test", X_bike_te.shape)

### 4.2 Four metrics, and what each one punishes

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_bike_te, y_bike_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_bike_te, y_bike_pred)
r2 = r2_score(y_bike_te, y_bike_pred)

print(f"MSE  {mse:10.2f}   squared rentals  — not interpretable on its own")
print(f"RMSE {rmse:10.2f}   rentals          — same units as the target")
print(f"MAE  {mae:10.2f}   rentals          — same units, less sensitive to large errors")
print(f"R^2  {r2:10.4f}   unitless         — share of variance explained")
print()
print(f"for reference, the mean hourly demand is {y_bike_te.mean():.2f} rentals")
print(f"so a typical prediction is off by roughly {rmse:.0f} rentals, about "
      f"{100 * rmse / y_bike_te.mean():.0f}% of the average hour")

**MSE** squares the errors, so an error of 100 counts a hundred times as
much as an error of 10. That makes it sensitive to a handful of bad hours,
and it is measured in squared rentals, which is not a quantity anyone has
an intuition for.

**RMSE** is the square root of MSE, which puts it back into rentals. This
is why it is the usual reporting metric: you can say "typically wrong by
about 100 bicycles" and be understood.

**MAE** averages the absolute errors. It treats an error of 100 as ten
times an error of 10, not a hundred times, so it is less influenced by a
few extreme hours.

**R²** is the share of the variance in `cnt` that the model explains, on a
scale where 1.0 is perfect and 0.0 is no better than always predicting the
mean. It is unitless, which makes it comparable across problems and
useless for saying how wrong any individual prediction is.

The difference between RMSE and MAE is itself diagnostic:

In [ ]:
errors = y_bike_te - y_bike_pred

print(f"RMSE {rmse:.2f}  MAE {mae:.2f}   ratio {rmse / mae:.3f}")
print()
print("RMSE exceeds MAE whenever errors are unevenly sized. A ratio near 1")
print("means uniform errors; a large ratio means a few hours are badly wrong.")
print()
print("worst 5 absolute errors:", np.sort(np.abs(errors))[-5:].round(1))
print("median absolute error  :", round(float(np.median(np.abs(errors))), 2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].scatter(y_bike_te, y_bike_pred, s=6, alpha=0.25)
limit = max(y_bike_te.max(), y_bike_pred.max())
axes[0].plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1.5,
             label="perfect prediction")
axes[0].set_xlabel("actual rentals")
axes[0].set_ylabel("predicted rentals")
axes[0].set_title(f"Predicted against actual (R^2 = {r2:.3f})")
axes[0].legend()

axes[1].scatter(y_bike_pred, errors, s=6, alpha=0.25)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("predicted rentals")
axes[1].set_ylabel("residual (actual - predicted)")
axes[1].set_title("Residuals")

fig.suptitle("A linear model on hourly bicycle demand")
fig.tight_layout()
plt.show()

Two failures are visible and neither shows up in R².

The model **predicts negative rentals**, which is impossible. Nothing in a
linear regression knows that a count cannot be below zero.

The residuals **fan out**: the spread of errors grows with the predicted
value. That is heteroscedasticity, and it violates the constant-variance
assumption noted when this data's synthetic cousin was built in Practice 1
§6.1.

In [ ]:
print("predictions below zero:", int((y_bike_pred < 0).sum()),
      f"of {len(y_bike_pred)} ({100 * (y_bike_pred < 0).mean():.1f}%)")
print("most negative prediction:", round(float(y_bike_pred.min()), 1), "rentals")
print()
print("error spread in the lowest and highest prediction deciles:")
low = errors[y_bike_pred <= np.quantile(y_bike_pred, 0.1)]
high = errors[y_bike_pred >= np.quantile(y_bike_pred, 0.9)]
print(f"  lowest decile  std {low.std():7.2f}")
print(f"  highest decile std {high.std():7.2f}")

A metric summarises; a residual plot diagnoses. Report both.

## 5. Train, validation and test

### 5.1 Why a held-out set exists at all

A model can always be made to fit the data it was shown. The question is
what it does on data it was not shown, and the only way to find out is to
keep some back.

In [ ]:
train_score = bike_model.score(X_bike_tr, y_bike_tr)
test_score = bike_model.score(X_bike_te, y_bike_te)

print(f"R^2 on training data: {train_score:.4f}")
print(f"R^2 on held-out data: {test_score:.4f}")
print(f"gap: {train_score - test_score:+.4f}")

For this model the gap is small, which is evidence that a linear model on
these features is not memorising. §7 builds a model where the gap is
enormous.

### 5.2 Three splits, not two

- **Training set** — the model learns its parameters here.
- **Validation set** — you compare models and choose hyperparameters here.
- **Test set** — you measure the chosen model here, **once**, at the end.

The reason for the middle one is that choosing is itself a form of
fitting. If you try thirty values of a hyperparameter and keep the one
that scored best on a set, that set has trained your choice, and its score
is no longer an unbiased estimate of anything.

§10.4 demonstrates this with real numbers on Titanic.

In [ ]:
X_demo = X_bikes[:4000]
y_demo = y_bikes[:4000]

# First split off the test set and set it aside.
X_rest, X_test_demo, y_rest, y_test_demo = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=SEED
)
# Then split what remains into train and validation.
X_train_demo, X_val_demo, y_train_demo, y_val_demo = train_test_split(
    X_rest, y_rest, test_size=0.25, random_state=SEED
)

print(f"train      {len(X_train_demo):5d}  ({len(X_train_demo)/len(X_demo):.0%})  fit parameters here")
print(f"validation {len(X_val_demo):5d}  ({len(X_val_demo)/len(X_demo):.0%})  choose between models here")
print(f"test       {len(X_test_demo):5d}  ({len(X_test_demo)/len(X_demo):.0%})  measure once, at the end")

### 5.3 `stratify`, and why classification needs it

A random split can give two halves with different class balances,
especially when a class is small. `stratify=y` forces the split to
preserve the proportions.

In [ ]:
titanic = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/titanic.csv"
)
y_demo_labels = titanic["Survived"].values

print("full dataset balance:", np.bincount(y_demo_labels) / len(y_demo_labels))
print()
for label, kwargs in [("without stratify", {}), ("with stratify", {"stratify": y_demo_labels})]:
    spread = []
    for seed in range(8):
        _, _, y_a, y_b = train_test_split(
            np.zeros((len(y_demo_labels), 1)), y_demo_labels,
            test_size=0.25, random_state=seed, **kwargs
        )
        spread.append(y_b.mean())
    print(f"{label:17} test-set survival rate over 8 seeds: "
          f"min {min(spread):.4f}  max {max(spread):.4f}  range {max(spread) - min(spread):.4f}")

In [ ]:
# LOCAL ALTERNATIVE. Run EITHER this cell OR the one above, not both.
# Above reads over the network and is what Colab needs. This one reads
# the same files from your checkout, for the dev-env container. The
# data is identical; only the address differs.
titanic = pd.read_csv("../datasets/titanic.csv")
y_demo_labels = titanic["Survived"].values

print("full dataset balance:", np.bincount(y_demo_labels) / len(y_demo_labels))
print()
for label, kwargs in [("without stratify", {}), ("with stratify", {"stratify": y_demo_labels})]:
    spread = []
    for seed in range(8):
        _, _, y_a, y_b = train_test_split(
            np.zeros((len(y_demo_labels), 1)), y_demo_labels,
            test_size=0.25, random_state=seed, **kwargs
        )
        spread.append(y_b.mean())
    print(f"{label:17} test-set survival rate over 8 seeds: "
          f"min {min(spread):.4f}  max {max(spread):.4f}  range {max(spread) - min(spread):.4f}")

Without stratification the test-set survival rate wanders by several
percentage points depending on the seed alone. Any model comparison whose
difference is smaller than that wander is measuring the split, not the
models.

### 5.4 There is a ceiling, and it is not 100 %

Rebuild the two-blob classification dataset from Practice 1 §6.2. The two
classes overlap by construction, so some points genuinely could have come
from either. No classifier can separate those.

In [ ]:
classification_rng = np.random.default_rng(SEED + 1)

mean_class_0, cov_class_0 = np.array([1.0, 2.0]), np.eye(2)
mean_class_1, cov_class_1 = np.array([4.0, 5.0]), 2.0 * np.eye(2)
n_per_class = 150

points_class_0 = classification_rng.multivariate_normal(mean_class_0, cov_class_0, n_per_class)
points_class_1 = classification_rng.multivariate_normal(mean_class_1, cov_class_1, n_per_class)

X_blobs = np.concatenate([points_class_0, points_class_1], axis=0)
y_blobs = np.concatenate([np.zeros(n_per_class, dtype=int), np.ones(n_per_class, dtype=int)])

print("X", X_blobs.shape, " y", y_blobs.shape, " balance", np.bincount(y_blobs))

Because both distributions are known exactly, the best achievable accuracy
can be computed rather than guessed. The optimal rule assigns each point to
whichever class had the higher probability density there; its error rate is
the **Bayes error**, and it is a property of the data, not of any model.

In [ ]:
from scipy.stats import multivariate_normal

density_0 = multivariate_normal(mean_class_0, cov_class_0)
density_1 = multivariate_normal(mean_class_1, cov_class_1)

# Equal priors: 150 points from each class.
optimal_prediction = (density_1.pdf(X_blobs) > density_0.pdf(X_blobs)).astype(int)
bayes_accuracy = (optimal_prediction == y_blobs).mean()

print(f"best possible accuracy on this sample: {bayes_accuracy:.4f}")
print(f"so the best possible error rate is     {1 - bayes_accuracy:.4f}")
print()
print("Any model scoring near this has extracted everything the data contains.")
print("A model scoring ABOVE it on the test set has got lucky, not got better.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

X_blob_tr, X_blob_te, y_blob_tr, y_blob_te = train_test_split(
    X_blobs, y_blobs, test_size=0.3, random_state=SEED, stratify=y_blobs
)

for name, model in [
    ("logistic regression", LogisticRegression(random_state=SEED)),
    ("linear SVM", SVC(kernel="linear", C=1.0, random_state=SEED)),
    ("decision tree", DecisionTreeClassifier(random_state=SEED)),
]:
    model.fit(X_blob_tr, y_blob_tr)
    print(f"{name:22} test accuracy {model.score(X_blob_te, y_blob_te):.4f}")

print(f"\n{'Bayes ceiling':22} {bayes_accuracy:.4f}")

Three very different models land within a few points of each other and of
the ceiling. That is the signature of a problem where the data, not the
algorithm, is the binding constraint — and it is much more common than
the literature suggests. When three model families agree, reach for better
features rather than a fourth model.

Now look again at the two linear models: **they scored above the
ceiling.** That is not an error in the ceiling and it is not a better
model. The Bayes accuracy is the expected performance of the optimal rule
over the whole distribution; the test score is one sample of 90 points.
With 90 points, one standard error on an accuracy near 0.96 is about
2 percentage points, so landing a point or two either side of the ceiling
is ordinary sampling variation.

The lesson is about how to read a test score in general. A single number
computed on a finite held-out sample is itself an estimate with a spread,
and differences smaller than that spread are not results. §6.2 replaces
the single number with a distribution across folds for exactly this
reason.

## 6. Cross-validation and the `Pipeline`

### 6.1 One split is one measurement

A single train/test split gives one number, and that number depends on
which rows happened to land where. §5.3 showed the split moving the class
balance; it moves the score too.

In [ ]:
wine = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/wine.csv"
)
wine_features = [c for c in wine.columns if c != "target"]
X_wine = wine[wine_features].values
y_wine = wine["target"].values

from sklearn.preprocessing import StandardScaler

scores_by_seed = []
for seed in range(10):
    X_a, X_b, y_a, y_b = train_test_split(
        X_wine, y_wine, test_size=0.3, random_state=seed, stratify=y_wine
    )
    scaler = StandardScaler().fit(X_a)
    model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED)
    model.fit(scaler.transform(X_a), y_a)
    scores_by_seed.append(model.score(scaler.transform(X_b), y_b))

scores_by_seed = np.array(scores_by_seed)
print("test accuracy over 10 different splits of the same data:")
print(" ", scores_by_seed.round(4))
print(f"\nmin {scores_by_seed.min():.4f}   max {scores_by_seed.max():.4f}"
      f"   spread {scores_by_seed.max() - scores_by_seed.min():.4f}")
print(f"mean {scores_by_seed.mean():.4f} +/- {scores_by_seed.std():.4f}")

In [ ]:
# LOCAL ALTERNATIVE. Run EITHER this cell OR the one above, not both.
# Above reads over the network and is what Colab needs. This one reads
# the same files from your checkout, for the dev-env container. The
# data is identical; only the address differs.
wine = pd.read_csv("../datasets/wine.csv")
wine_features = [c for c in wine.columns if c != "target"]
X_wine = wine[wine_features].values
y_wine = wine["target"].values

from sklearn.preprocessing import StandardScaler

scores_by_seed = []
for seed in range(10):
    X_a, X_b, y_a, y_b = train_test_split(
        X_wine, y_wine, test_size=0.3, random_state=seed, stratify=y_wine
    )
    scaler = StandardScaler().fit(X_a)
    model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED)
    model.fit(scaler.transform(X_a), y_a)
    scores_by_seed.append(model.score(scaler.transform(X_b), y_b))

scores_by_seed = np.array(scores_by_seed)
print("test accuracy over 10 different splits of the same data:")
print(" ", scores_by_seed.round(4))
print(f"\nmin {scores_by_seed.min():.4f}   max {scores_by_seed.max():.4f}"
      f"   spread {scores_by_seed.max() - scores_by_seed.min():.4f}")
print(f"mean {scores_by_seed.mean():.4f} +/- {scores_by_seed.std():.4f}")

The same model on the same data scores anywhere in a band several points
wide, depending only on `random_state`. Reporting one of those numbers as
"the accuracy" is reporting a draw from a distribution as if it were a
constant.

### 6.2 Cross-validation reports the distribution

k-fold cross-validation splits the data into `k` parts, trains on `k - 1`
and tests on the remaining one, and rotates. Every row is used for testing
exactly once.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline

pipeline = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_scores = cross_val_score(pipeline, X_wine, y_wine, cv=cv, scoring="accuracy")

print("accuracy per fold:", fold_scores.round(4))
print(f"mean {fold_scores.mean():.4f}   std {fold_scores.std():.4f}")
print(f"report it as: {fold_scores.mean():.3f} +/- {fold_scores.std():.3f}")

Read the spread, not only the mean. A model at 0.97 ± 0.01 and one at
0.97 ± 0.06 are not equally trustworthy, and the second may simply be
unstable on this much data.

**`KFold` against `StratifiedKFold`.** Plain `KFold` cuts the data into
contiguous chunks after an optional shuffle; if the classes are unevenly
distributed, a fold can end up missing one. `StratifiedKFold` preserves
the class proportions in every fold, and it is the right default for
classification.

In [ ]:
# An ordered target makes the failure obvious: wine.csv is sorted by cultivar.
print("first and last 10 targets in file order:", y_wine[:10], "...", y_wine[-10:])
print()

plain = KFold(n_splits=3, shuffle=False)
print("KFold without shuffling, class counts in each test fold:")
for i, (_, test_index) in enumerate(plain.split(X_wine, y_wine)):
    print(f"  fold {i}: {np.bincount(y_wine[test_index], minlength=3)}")

print("\nStratifiedKFold, class counts in each test fold:")
for i, (_, test_index) in enumerate(StratifiedKFold(n_splits=3).split(X_wine, y_wine)):
    print(f"  fold {i}: {np.bincount(y_wine[test_index], minlength=3)}")

In [ ]:
print("accuracy with unshuffled KFold on this sorted file:")
print(" ", cross_val_score(pipeline, X_wine, y_wine, cv=KFold(n_splits=3)).round(4))
print("\naccuracy with StratifiedKFold:")
print(" ", cross_val_score(pipeline, X_wine, y_wine, cv=StratifiedKFold(n_splits=3)).round(4))

Each unshuffled fold tests almost entirely on a class the model barely
saw, and the accuracy collapses. This is not a subtle effect and it is
produced entirely by the order of rows in a file.

### 6.3 Where the scaler has to live

Practice 2 §14.3 established that a scaler is fitted on training data only.
Cross-validation makes that harder than it looks, because "the training
data" is different in every fold.

Here is the mistake, written out. Scale once, then cross-validate:

In [ ]:
# WRONG. The scaler sees every row, including the rows that will serve as
# each fold's validation set.
X_wine_prescaled = StandardScaler().fit_transform(X_wine)
leaky_scores = cross_val_score(
    SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED),
    X_wine_prescaled, y_wine, cv=cv, scoring="accuracy",
)

# CORRECT. The scaler is a step of the pipeline, so it is re-fitted inside
# every fold on that fold's training rows only.
clean_scores = cross_val_score(pipeline, X_wine, y_wine, cv=cv, scoring="accuracy")

print(f"scaler outside the pipeline (leaks): {leaky_scores.mean():.4f} +/- {leaky_scores.std():.4f}")
print(f"scaler inside the pipeline (correct): {clean_scores.mean():.4f} +/- {clean_scores.std():.4f}")
print(f"difference: {leaky_scores.mean() - clean_scores.mean():+.4f}")

The two numbers are identical, and that is worth explaining rather than
hiding. `StandardScaler` learns only a mean and a standard deviation per
column. On 178 well-behaved rows those two numbers barely move when a
fifth of the rows is held out, so the leak exists but is far below the
resolution of the score.

**The size of the leak scales with how much the preprocessing step learns
from the data.** A scaler learns 26 numbers. A feature selector choosing
10 columns out of 2000 makes a decision with an enormous number of
possible outcomes, and it makes that decision by looking at the
relationship between the features and the target — the very thing the
model is supposed to discover.

Here is that case, on data containing **no signal at all**: 100 samples,
2000 features of pure noise, and labels that are coin flips. Nothing can
predict anything here, and the honest accuracy is 0.5.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

noise_rng = np.random.default_rng(SEED)
n_samples, n_features = 100, 2000

X_noise = noise_rng.normal(size=(n_samples, n_features))   # pure noise
y_noise = noise_rng.integers(0, 2, n_samples)              # coin flips, unrelated to X

print("features are noise, labels are coin flips, so nothing is predictable.")
print("the only honest accuracy on this data is 0.5000")

In [ ]:
# WRONG. Choose the 10 "best" features using every row, then cross-validate.
selector_all = SelectKBest(f_classif, k=10).fit(X_noise, y_noise)
X_preselected = selector_all.transform(X_noise)
leaky_selection = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression(random_state=SEED)),
    X_preselected, y_noise, cv=cv,
)

# CORRECT. The selector is a pipeline step, re-fitted inside every fold.
clean_selection = cross_val_score(
    make_pipeline(SelectKBest(f_classif, k=10), StandardScaler(),
                  LogisticRegression(random_state=SEED)),
    X_noise, y_noise, cv=cv,
)

print(f"selection outside the pipeline (leaks) : {leaky_selection.mean():.4f}")
print(f"selection inside the pipeline (correct): {clean_selection.mean():.4f}")
print(f"the truth                              : 0.5000")

The leaked procedure reports an accuracy far above chance **on data that
contains no information**. It is not a subtle bias; it is an entirely
invented result, and it would survive peer review in any write-up that
only said "we used 5-fold cross-validation".

The mechanism: with 2000 noise features and 100 rows, some features
correlate with the labels by chance. Choosing them using all 100 rows
means the chance correlation is present in every validation fold as well,
so the model "confirms" it. Inside the pipeline, the selection is made
from the training rows of each fold and the chance correlations it picks
up do not recur in that fold's held-out rows.

**The rule, stated once: every step that learns anything from data belongs
inside the `Pipeline`.** Not for tidiness — so that cross-validation can
re-fit it on each fold's training rows, which is the only way the
resulting score estimates anything real.

A `Pipeline` is itself an estimator. It has `fit`, `predict` and `score`,
and it can be passed anywhere a model can:

In [ ]:
print("pipeline steps:", [name for name, _ in pipeline.steps])
print("is it an estimator?", hasattr(pipeline, "fit"), hasattr(pipeline, "predict"))
print()
pipeline.fit(X_wine, y_wine)
print("reaching a step by name:", pipeline.named_steps["svc"].kernel)
print("the scaler's learned means (first 3):", pipeline.named_steps["standardscaler"].mean_[:3].round(3))

### 6.4 Tuning hyperparameters over the pipeline

`GridSearchCV` tries every combination of the values you list, scoring
each by cross-validation, and refits the best on the whole training set.

Parameters are addressed as `stepname__parametername`, with a double
underscore.

In [ ]:
from sklearn.model_selection import GridSearchCV

X_wine_tr, X_wine_te, y_wine_tr, y_wine_te = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=SEED, stratify=y_wine
)

# The drafts searched C in {0.1, 1, 10, 100, 1000} x gamma in {1, 0.1, 0.01,
# 0.001, 0.0001} = 25 combinations x 5 folds = 125 fits. That is kept here in
# full: on 124 training rows it costs well under a second.
grid = {
    "svc__C": [0.1, 1, 10, 100, 1000],
    "svc__gamma": [1, 0.1, 0.01, 0.001, 0.0001],
    "svc__kernel": ["rbf"],
}

search = GridSearchCV(
    make_pipeline(StandardScaler(), SVC(random_state=SEED)),
    param_grid=grid, cv=cv, scoring="accuracy", refit=True, n_jobs=1,
)
search.fit(X_wine_tr, y_wine_tr)

print("combinations tried:", len(search.cv_results_["params"]))
print("best parameters   :", search.best_params_)
print(f"best CV accuracy  : {search.best_score_:.4f}")
print(f"held-out test accuracy of the refitted model: {search.score(X_wine_te, y_wine_te):.4f}")

`refit=True` means `search` is now the best model, fitted on all the
training data, ready to predict. `best_score_` is the cross-validated
score of that configuration — it is **not** a test score, because the
configuration was chosen using it.

The test score on the last line is the honest one, and it is computed
once.

In [ ]:
results = pd.DataFrame(search.cv_results_)
heat = results.pivot_table(index="param_svc__gamma", columns="param_svc__C",
                           values="mean_test_score")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(heat, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("Cross-validated accuracy over the C x gamma grid")
ax.set_xlabel("C")
ax.set_ylabel("gamma")
plt.show()

The surface is flat over a broad region and falls off a cliff at large
`gamma`. That shape is worth more than the single best cell: it says the
model is insensitive to `C` across three orders of magnitude here, so
reporting `C = 10` as "the optimal value" would overstate what the data
supports.

### 6.5 Choosing the number of components by the task

Practice 2 §15.3 left the number of PCA components as an open question and
promised this section. A component count is just another hyperparameter,
so put PCA in the pipeline and tune it.

In [ ]:
from sklearn.decomposition import PCA

pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(random_state=SEED)),
    ("svc", SVC(kernel="rbf", C=10, gamma="scale", random_state=SEED)),
])

component_search = GridSearchCV(
    pca_pipeline, param_grid={"pca__n_components": [2, 3, 4, 5, 6, 8, 10, 13]},
    cv=cv, scoring="accuracy",
)
component_search.fit(X_wine_tr, y_wine_tr)

for n, score in zip(component_search.cv_results_["param_pca__n_components"],
                    component_search.cv_results_["mean_test_score"]):
    print(f"  n_components = {n:2}  CV accuracy {score:.4f}")
print("\nbest:", component_search.best_params_, f"at {component_search.best_score_:.4f}")

That is an answer derived from the task rather than from a variance
threshold. Note that it need not agree with the 95 %-of-variance rule:
variance is not the same thing as usefulness for a particular
classification, and a low-variance direction can be the one that separates
two classes.

## 7. Polynomial regression, and the bias-variance trade-off

### 7.1 Fitting a curve with a linear model

A "linear model" is linear in its **coefficients**, not in its inputs.
Expand `x` into `x, x², x³, ...` and a straight-line fitter draws curves.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_rng = np.random.default_rng(SEED)
n_poly = 50
poly_noise_sigma = 12.0

x_poly = poly_rng.uniform(-3, 3, n_poly)
true_curve = lambda t: t ** 4 - 3 * t ** 2 + t + 5
y_poly = true_curve(x_poly) + poly_rng.normal(0, poly_noise_sigma, n_poly)

X_poly = x_poly.reshape(-1, 1)
x_dense = np.linspace(-3, 3, 400).reshape(-1, 1)

# A small sample and substantial noise, on purpose. With 50 points there is
# not enough evidence to support a very flexible model, which is what makes
# the overfitting in section 7.3 visible rather than marginal.
print("true function: y = x^4 - 3x^2 + x + 5")
print(f"plus normal(0, {poly_noise_sigma}) noise, n = {n_poly}")
print("true function ranges from about",
      round(float(true_curve(x_dense.ravel()).min()), 1), "to",
      round(float(true_curve(x_dense.ravel()).max()), 1))

In [ ]:
degree_4_model = make_pipeline(PolynomialFeatures(degree=4), LinearRegression())
degree_4_model.fit(X_poly, y_poly)

fig, ax = plt.subplots()
ax.scatter(x_poly, y_poly, s=22, alpha=0.6, label="noisy observations")
ax.plot(x_dense, true_curve(x_dense.ravel()), color="black", linewidth=2.5,
        label="true function")
ax.plot(x_dense, degree_4_model.predict(x_dense), color="tab:red", linestyle="--",
        linewidth=2.5, label="fitted, degree 4")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Polynomial regression: a linear model on expanded features")
ax.legend()
plt.show()

In [ ]:
linear_step = degree_4_model.named_steps["linearregression"]
sklearn_coefficients = np.hstack([linear_step.intercept_, linear_step.coef_[1:]])

print("recovered coefficients, LOWEST power first (scikit-learn's order):")
print("  ", sklearn_coefficients.round(4))
print("true                     [5, 1, -3, 0, 1]")

Compare those term by term. The `x^4` coefficient is recovered well and
the constant term is not close at all. With this much noise on 50 points
the individual coefficients are poorly determined even though, as the
figure above shows, **the fitted curve is a good description of the
data**.

That gap between "the model predicts well" and "the model's parameters
are trustworthy" is worth holding on to. They are different claims,
requiring different amounts of evidence, and a model can support the
first while entirely failing to support the second.

### 7.2 `np.polyfit` orders its coefficients the other way round

The same fit through NumPy. The result is identical mathematically and
**reversed** in presentation, which is a genuine source of error when
moving between the two.

In [ ]:
polyfit_coefficients = np.polyfit(x_poly, y_poly, deg=4)

print("np.polyfit returns HIGHEST power first:")
print("  ", polyfit_coefficients.round(4))
print("scikit-learn gave  LOWEST power first:")
print("  ", sklearn_coefficients.round(4))
print()
print("reversed, they agree:", np.allclose(polyfit_coefficients[::-1], sklearn_coefficients, atol=1e-6))
print()
print("np.polyval evaluates a polyfit result, respecting that order:")
print("  polyval at x = 2:", round(float(np.polyval(polyfit_coefficients, 2.0)), 4))
print("  pipeline at x = 2:", round(float(degree_4_model.predict([[2.0]])[0]), 4))

### 7.3 The degree is not known in advance

The draft this replaces set `degree=4` because the true function was
quartic. In real work nobody tells you the degree. Sweep it and watch
what happens on data the model has not seen.

In [ ]:
from sklearn.model_selection import cross_val_score

degrees = range(1, 16)
train_errors, validation_errors = [], []

for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
    model.fit(X_poly, y_poly)
    train_errors.append(mean_squared_error(y_poly, model.predict(X_poly)))
    cv_mse = -cross_val_score(model, X_poly, y_poly, cv=KFold(5, shuffle=True, random_state=SEED),
                              scoring="neg_mean_squared_error")
    validation_errors.append(cv_mse.mean())

train_errors = np.array(train_errors)
validation_errors = np.array(validation_errors)

print(f"{'degree':>7} {'train MSE':>12} {'validation MSE':>16}")
for degree, tr, va in zip(degrees, train_errors, validation_errors):
    marker = "  <- best validation" if va == validation_errors.min() else ""
    print(f"{degree:7} {tr:12.3f} {va:16.3f}{marker}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(list(degrees), train_errors, marker="o", label="training error")
ax.plot(list(degrees), validation_errors, marker="s", label="validation error (5-fold)")
ax.axvline(4, color="grey", linestyle=":", label="true degree = 4")
ax.set_yscale("log")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("mean squared error (log scale)")
ax.set_title("Bias-variance: training error always falls, validation error does not")
ax.set_xticks(list(degrees))
ax.legend()
plt.show()

Three regions, and they are the whole of the bias-variance trade-off.

**Degrees 1-3: underfitting, high bias.** The model is too rigid to
represent a quartic. Training and validation error are both high and close
together — the model is not memorising, it is incapable.

**Degree 4-5: the sweet spot.** Validation error is at its minimum.

**Degrees 10-15: overfitting, high variance.** Training error keeps
falling, because a higher-degree polynomial can always pass closer to the
points it was given. Validation error climbs, because the extra
flexibility is spent reproducing noise that will not recur.

**The gap between the two curves is the diagnosis.** Both high and close:
underfitting, get a more flexible model. Training low and validation much
higher: overfitting, get more data or less flexibility.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, degree in zip(axes, [1, 4, 15]):
    model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
    model.fit(X_poly, y_poly)
    ax.scatter(x_poly, y_poly, s=18, alpha=0.5)
    ax.plot(x_dense, true_curve(x_dense.ravel()), color="black", linewidth=2, label="true")
    ax.plot(x_dense, model.predict(x_dense), color="tab:red", linestyle="--",
            linewidth=2, label=f"degree {degree}")
    ax.set_title(f"degree {degree}: validation MSE {validation_errors[degree - 1]:.1f}")
    ax.set_xlabel("x")
    ax.set_ylim(-35, 95)
    ax.legend(fontsize=8)
axes[0].set_ylabel("y")

fig.suptitle("Underfitting, about right, and overfitting")
fig.tight_layout()
plt.show()

The degree-15 curve on the right is the wiggle promised in Practice 1 §5.3.
It passes nearer the training points than the true function does, and it
is a worse description of the process that generated them — visibly so
near the edges, where it swings away with nothing to constrain it.

## 8. Decision trees

### 8.1 A tree on Iris

A decision tree asks a sequence of yes/no questions about one feature at
a time. It is the one model in this course you can read directly.

In [ ]:
iris = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/iris.csv"
)

X_iris = iris.drop(columns=["Id", "Species"])
y_iris = iris["Species"]

print("X is a", type(X_iris).__name__, X_iris.shape)
print("y is a", type(y_iris).__name__, y_iris.shape)
print("classes:", sorted(set(y_iris)))
print("features:", list(X_iris.columns))

In [ ]:
# LOCAL ALTERNATIVE. Run EITHER this cell OR the one above, not both.
# Above reads over the network and is what Colab needs. This one reads
# the same files from your checkout, for the dev-env container. The
# data is identical; only the address differs.
iris = pd.read_csv("../datasets/iris.csv")

X_iris = iris.drop(columns=["Id", "Species"])
y_iris = iris["Species"]

print("X is a", type(X_iris).__name__, X_iris.shape)
print("y is a", type(y_iris).__name__, y_iris.shape)
print("classes:", sorted(set(y_iris)))
print("features:", list(X_iris.columns))

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

X_iris_tr, X_iris_te, y_iris_tr, y_iris_te = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=SEED, stratify=y_iris
)

tree = DecisionTreeClassifier(criterion="gini", random_state=SEED)
tree.fit(X_iris_tr, y_iris_tr)

print(f"training accuracy {tree.score(X_iris_tr, y_iris_tr):.4f}")
print(f"test accuracy     {tree.score(X_iris_te, y_iris_te):.4f}")
print(f"depth reached     {tree.get_depth()}   leaves {tree.get_n_leaves()}")

Training accuracy is 1.0. An unconstrained tree splits until every leaf
is pure, so it will always reproduce its training data exactly. That
number carries no information about the model at all — §8.3 is about what
it costs.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_tree(tree, feature_names=list(X_iris.columns), class_names=sorted(set(y_iris)),
          filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title("A decision tree grown to purity on Iris")
plt.show()

Read the root. The first split is on a petal measurement, which is what
Practice 2 §11.3's pairplot predicted: petal length and width separate
setosa completely, and the tree found the same structure by a different
route.

Each node shows the splitting rule, the impurity, the number of samples
reaching it, and the class distribution there.

### 8.2 Gini and entropy

Both measure how mixed a node is. Both are zero for a pure node and
maximal for an even split. `gini` is slightly cheaper to compute;
`entropy` comes from information theory.

In [ ]:
def gini(proportions):
    return 1 - np.sum(np.square(proportions))

def entropy(proportions):
    nonzero = proportions[proportions > 0]
    return -np.sum(nonzero * np.log2(nonzero))

print(f"{'class split':>22} {'gini':>8} {'entropy':>9}")
for label, proportions in [
    ("pure [1, 0, 0]", np.array([1.0, 0.0, 0.0])),
    ("nearly pure [.9,.1,0]", np.array([0.9, 0.1, 0.0])),
    ("two-way even [.5,.5,0]", np.array([0.5, 0.5, 0.0])),
    ("three-way even", np.array([1/3, 1/3, 1/3])),
]:
    print(f"{label:>22} {gini(proportions):8.4f} {entropy(proportions):9.4f}")

In [ ]:
for criterion in ("gini", "entropy"):
    scores = cross_val_score(
        DecisionTreeClassifier(criterion=criterion, random_state=SEED),
        X_iris, y_iris, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    )
    print(f"{criterion:8} CV accuracy {scores.mean():.4f} +/- {scores.std():.4f}")

They agree, which is the usual outcome. The criterion is rarely the
hyperparameter that matters; the next one is.

### 8.3 Depth, pruning, and what purity costs

An unconstrained tree memorises. `max_depth` and `min_samples_leaf` limit
how far it can go.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, None]
rows = []
inner_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    model.fit(X_iris_tr, y_iris_tr)
    cv_scores = cross_val_score(model, X_iris_tr, y_iris_tr, cv=inner_cv)
    rows.append({
        "max_depth": "None" if depth is None else depth,
        "train": model.score(X_iris_tr, y_iris_tr),
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "leaves": model.get_n_leaves(),
    })

depth_table = pd.DataFrame(rows)
print(depth_table.to_string(index=False))

Training accuracy rises to 1.0 and stays there. Cross-validated accuracy
peaks early and then flattens or dips. The extra depth buys leaves, not
generalisation: a tree with more leaves than it needs has carved the
feature space into regions justified by one or two training points each.

In [ ]:
shallow = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_iris_tr, y_iris_tr)

fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(shallow, feature_names=list(X_iris.columns), class_names=sorted(set(y_iris)),
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title(f"Pruned to max_depth=3: {shallow.get_n_leaves()} leaves, "
             f"test accuracy {shallow.score(X_iris_te, y_iris_te):.4f}")
plt.show()

print(f"grown to purity: {tree.get_n_leaves()} leaves, test accuracy {tree.score(X_iris_te, y_iris_te):.4f}")
print(f"pruned to depth 3: {shallow.get_n_leaves()} leaves, test accuracy {shallow.score(X_iris_te, y_iris_te):.4f}")

A smaller tree, the same or better test accuracy, and a model a domain
expert can read in one sitting. `min_samples_leaf` achieves the same thing
from the other direction — it refuses to create a leaf backed by fewer
than `n` samples:

In [ ]:
for min_leaf in (1, 2, 5, 10, 20):
    model = DecisionTreeClassifier(min_samples_leaf=min_leaf, random_state=SEED).fit(X_iris_tr, y_iris_tr)
    cv_scores = cross_val_score(model, X_iris_tr, y_iris_tr, cv=inner_cv)
    print(f"min_samples_leaf = {min_leaf:2}   leaves {model.get_n_leaves():2}"
          f"   train {model.score(X_iris_tr, y_iris_tr):.4f}   CV {cv_scores.mean():.4f}")

### 8.4 Trees do not care about scale

The claim from Practice 2 §14.2, tested. A tree splits on thresholds of one
feature at a time, so any monotonic rescaling of a feature moves the
threshold and changes nothing else.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Multiply the LEAST informative feature by 1000. Lesson 2 section 11.3
# showed that sepal width separates the species worst; blowing up its scale
# makes it dominate any Euclidean distance.
X_iris_distorted = X_iris.copy()
X_iris_distorted["SepalWidthCm"] = X_iris_distorted["SepalWidthCm"] * 1000

tree_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
original = cross_val_score(DecisionTreeClassifier(random_state=SEED), X_iris, y_iris, cv=tree_cv)
distorted = cross_val_score(DecisionTreeClassifier(random_state=SEED), X_iris_distorted, y_iris, cv=tree_cv)
knn_original = cross_val_score(KNeighborsClassifier(), X_iris, y_iris, cv=tree_cv)
knn_distorted = cross_val_score(KNeighborsClassifier(), X_iris_distorted, y_iris, cv=tree_cv)

print("sepal width — the weakest feature — multiplied by 1000:")
print(f"  tree, original scales : {original.mean():.4f}")
print(f"  tree, distorted scales: {distorted.mean():.4f}   <- identical")
print(f"  KNN,  original scales : {knn_original.mean():.4f}")
print(f"  KNN,  distorted scales: {knn_distorted.mean():.4f}   <- collapsed")

The tree is unmoved to four decimal places. KNN loses nearly twenty
points, because Euclidean distance is now almost entirely a measurement
of sepal width, and sepal width is the feature that discriminates worst.

Note which feature was amplified. Blowing up the *most* informative
feature instead would barely hurt KNN, and could even help it — the
damage comes from letting an uninformative feature dominate the metric.
Since you do not know in advance which features are informative,
standardising is the only safe default for any distance-based model.

This is the whole content of "which models need scaling", demonstrated
rather than asserted.

## 9. Classification metrics

### 9.1 The confusion matrix

Accuracy is one number and it hides which mistakes were made. The
confusion matrix does not.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

iris_predictions = shallow.predict(X_iris_te)
matrix = confusion_matrix(y_iris_te, iris_predictions)

print("rows = actual, columns = predicted")
print(pd.DataFrame(matrix,
                   index=[f"actual {c}" for c in shallow.classes_],
                   columns=[f"pred {c}" for c in shallow.classes_]))

### 9.2 The two-class case, read cell by cell

For a binary problem the matrix has four cells and each has a name. This
is the worked reading carried over from the drafts, applied to a fresh
Titanic model.

In [ ]:
titanic_binary = titanic.copy()
titanic_binary["Age"] = titanic_binary["Age"].fillna(
    titanic_binary.groupby(["Pclass", "Sex"])["Age"].transform("median")
)
titanic_binary["Sex"] = (titanic_binary["Sex"] == "female").astype(int)
simple_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]

X_simple = titanic_binary[simple_features].values
y_simple = titanic_binary["Survived"].values

X_s_tr, X_s_te, y_s_tr, y_s_te = train_test_split(
    X_simple, y_simple, test_size=0.2, random_state=SEED, stratify=y_simple
)

simple_tree = DecisionTreeClassifier(max_depth=4, random_state=SEED).fit(X_s_tr, y_s_tr)
y_s_pred = simple_tree.predict(X_s_te)

cm = confusion_matrix(y_s_te, y_s_pred)
tn, fp, fn, tp = cm.ravel()

print(pd.DataFrame(cm, index=["actual died", "actual survived"],
                   columns=["pred died", "pred survived"]))
print()
print(f"TN = {tn:3}  true negatives  — died, predicted died")
print(f"FP = {fp:3}  false positives — died, predicted survived   (a false alarm)")
print(f"FN = {fn:3}  false negatives — survived, predicted died   (a missed positive)")
print(f"TP = {tp:3}  true positives  — survived, predicted survived")

The two errors are not interchangeable and which one matters is a
question about the application, never about the data. A false alarm
sends a lifeboat to someone who did not need it; a missed positive leaves
someone behind. In a medical screen the asymmetry is starker still.

### 9.3 Precision, recall and F1, built by hand

Compute them from the four cells before calling the library, so the
numbers are not magic.

In [ ]:
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
accuracy = (tp + tn) / (tp + tn + fp + fn)
specificity = tn / (tn + fp)

print(f"accuracy    = (TP + TN) / total        = {accuracy:.4f}")
print(f"precision   = TP / (TP + FP)           = {precision:.4f}")
print(f"  of those predicted to survive, this fraction did")
print(f"recall      = TP / (TP + FN)           = {recall:.4f}")
print(f"  of those who actually survived, this fraction was found")
print(f"specificity = TN / (TN + FP)           = {specificity:.4f}")
print(f"F1          = harmonic mean of the two = {f1:.4f}")

In [ ]:
print("the library agrees:")
print(classification_report(y_s_te, y_s_pred, target_names=["died", "survived"], digits=4))

`support` is the number of true instances of each class. `macro avg`
averages the per-class scores equally; `weighted avg` weights them by
support. On an imbalanced problem those two differ, and quoting the
flattering one is a standard way to overstate a result.

Precision and recall trade against each other. A model that predicts
"survived" for everyone has recall 1.0 and poor precision; one that
predicts it only when certain has high precision and poor recall. F1 is
their harmonic mean, which — unlike the arithmetic mean — stays low unless
both are high.

In [ ]:
print("why the HARMONIC mean:")
for p, r in [(1.0, 0.1), (0.9, 0.2), (0.55, 0.55)]:
    print(f"  precision {p:.2f}, recall {r:.2f}:  arithmetic {(p + r) / 2:.4f}"
          f"   harmonic (F1) {2 * p * r / (p + r):.4f}")
print("\nThe arithmetic mean rewards being excellent at one and useless at the")
print("other. The harmonic mean does not.")

### 9.4 Class imbalance and the baseline you must state first

Titanic is about 62 % died, 38 % survived (Practice 2 §3.4). So a model that
predicts "died" unconditionally is already 62 % accurate while having
learned nothing whatsoever.

**Always state the majority-class baseline before reporting an accuracy.**

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_s_tr, y_s_tr)
baseline_accuracy = baseline.score(X_s_te, y_s_te)

print(f"majority-class baseline accuracy: {baseline_accuracy:.4f}")
print(f"decision tree accuracy          : {accuracy_score(y_s_te, y_s_pred):.4f}")
print(f"improvement over doing nothing  : {accuracy_score(y_s_te, y_s_pred) - baseline_accuracy:+.4f}")
print()
print("the baseline's confusion matrix — it never predicts the positive class:")
print(confusion_matrix(y_s_te, baseline.predict(X_s_te)))
print()
print("and its recall on survivors is therefore exactly zero:")
print(classification_report(y_s_te, baseline.predict(X_s_te),
                            target_names=["died", "survived"],
                            digits=4, zero_division=0))

62 % accuracy, zero recall on the class anyone actually cares about. An
accuracy figure quoted without its baseline is not a result; it is a
statement about the class balance.

This is worse on rarer targets. At 1 % prevalence — fraud, a rare disease
— the do-nothing model is 99 % accurate, and accuracy has stopped being a
measurement at all.

In [ ]:
for prevalence in (0.50, 0.38, 0.10, 0.01):
    print(f"positive class at {prevalence:5.0%}  ->  always-predict-negative scores "
          f"{1 - prevalence:.1%} accuracy, with 0% recall")

## 10. K nearest neighbours on Titanic, end to end

KNN stores the training set and classifies a new point by a vote among
its `k` closest neighbours. It has no parameters to learn, which makes
`fit` trivial and `predict` expensive, and it makes the **distance
metric** the entire model. Everything that distorts distance — an
unscaled feature (§8.4), an invented ordering (Practice 2 §13.1) — attacks
KNN directly.

### 10.1 Preparing the table

Follow Practice 2 §16.1, but build the encoding **both ways** so the two can
be compared.

In [ ]:
def prepare_titanic(encoding):
    """Clean Titanic and encode Sex and Embarked either 'label' or 'onehot'."""
    frame = titanic.copy()
    frame = frame.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
    frame["Age"] = frame["Age"].fillna(
        frame.groupby(["Pclass", "Sex"])["Age"].transform("median")
    )
    frame["Embarked"] = frame["Embarked"].fillna(frame["Embarked"].mode()[0])

    if encoding == "label":
        # WRONG for a nominal feature: invents C < Q < S. See Lesson 2 section 13.1.
        frame["Sex"] = frame["Sex"].map({"male": 0, "female": 1})
        frame["Embarked"] = frame["Embarked"].map({"C": 0, "Q": 1, "S": 2})
    elif encoding == "onehot":
        frame["Sex"] = (frame["Sex"] == "female").astype(int)   # binary: no ordering possible
        frame = pd.get_dummies(frame, columns=["Embarked"], prefix="embarked", dtype=int)
    else:
        raise ValueError("encoding must be 'label' or 'onehot'")

    target = frame.pop("Survived").values
    return frame.values.astype(float), target, list(frame.columns)

X_label, y_titanic, label_columns = prepare_titanic("label")
X_onehot, _, onehot_columns = prepare_titanic("onehot")

print("label encoding :", X_label.shape, label_columns)
print("one-hot encoding:", X_onehot.shape, onehot_columns)

### 10.2 The same model, two encodings

Scale inside a pipeline, cross-validate, compare.

In [ ]:
knn_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

def knn_score(X, y, n_neighbors=5):
    pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=n_neighbors))
    scores = cross_val_score(pipeline, X, y, cv=knn_cv, scoring="accuracy")
    return scores.mean(), scores.std()

label_mean, label_std = knn_score(X_label, y_titanic)
onehot_mean, onehot_std = knn_score(X_onehot, y_titanic)

print(f"label encoding   (Embarked as 0/1/2): {label_mean:.4f} +/- {label_std:.4f}")
print(f"one-hot encoding (three columns)    : {onehot_mean:.4f} +/- {onehot_std:.4f}")
print(f"difference: {onehot_mean - label_mean:+.4f}")

**There is essentially no difference, and that is the honest result.**
It is smaller than the fold-to-fold standard deviation, so on this data
the two encodings are indistinguishable.

Do not conclude that the encoding does not matter. Conclude that it did
not matter *here*, and ask why. `Embarked` has **three** categories and
is a weak predictor of survival, so the fake ordering it introduces
distorts a small part of a distance that was not carrying much
information anyway. The damage from a fake ordering scales with two
things: how many categories there are, and how much the variable
actually matters.

Both of those are large for the hour-of-day column in the bicycle data.
It has 24 categories, it is the strongest single predictor of demand
(Practice 2 §8.4), and its relationship with demand is emphatically not
monotonic — there are peaks at 08:00 and 17:00 and a trough between them.
Integer encoding also claims that hour 23 and hour 0 are 23 units apart,
when they are adjacent.

`KNeighborsRegressor` is the regression counterpart of the classifier:
same idea, but it averages its neighbours' target values instead of
taking a vote.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

bike_subset = bikes.iloc[:4000]
bike_numeric = bike_subset[["temp", "hum", "windspeed"]].values
bike_target = bike_subset["cnt"].values.astype(float)

# hour as a single integer column: claims 23 is far from 0, and that the
# relationship between hour and demand is monotonic. Both are false.
X_hour_integer = np.hstack([bike_numeric, bike_subset[["hr"]].values])

# hour one-hot: 24 columns, every hour equidistant from every other.
X_hour_onehot = np.hstack([
    bike_numeric,
    pd.get_dummies(bike_subset["hr"].astype("category"), dtype=int).values,
])

regression_cv = KFold(5, shuffle=True, random_state=SEED)
for name, X in [("hour as an integer", X_hour_integer), ("hour one-hot", X_hour_onehot)]:
    scores = cross_val_score(
        make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)),
        X, bike_target, cv=regression_cv, scoring="r2",
    )
    print(f"  {name:20} R^2 {scores.mean():.4f} +/- {scores.std():.4f}")

Seven points of R², from nothing but the representation of one column.
The model, the data and the hyperparameters are identical; only the claim
about what "close together" means has changed.

So the rule stands, and the reason to follow it is not that it always
helps measurably. It is that **you cannot tell in advance** whether a
nominal variable is the harmless `Embarked` case or the expensive `hr`
case, and one-hot encoding costs you nothing but columns.

### 10.3 Sweeping K — the way the draft did it

Try every `k` from 1 to 30, score each **on the test set**, and keep the
best. This is what the source material did, and it is instructive.

In [ ]:
X_knn_tr, X_knn_te, y_knn_tr, y_knn_te = train_test_split(
    X_onehot, y_titanic, test_size=0.25, random_state=SEED, stratify=y_titanic
)

knn_scaler = StandardScaler().fit(X_knn_tr)
X_knn_tr_scaled = knn_scaler.transform(X_knn_tr)
X_knn_te_scaled = knn_scaler.transform(X_knn_te)

k_values = range(1, 31)
test_error_rates = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_knn_tr_scaled, y_knn_tr)
    test_error_rates.append(1 - model.score(X_knn_te_scaled, y_knn_te))

test_error_rates = np.array(test_error_rates)
best_k_on_test = int(np.argmin(test_error_rates)) + 1

print(f"best k by test error: {best_k_on_test}")
print(f"test accuracy there : {1 - test_error_rates.min():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(list(k_values), test_error_rates, marker="o", label="error rate on the test set")
ax.axvline(best_k_on_test, color="tab:red", linestyle=":", label=f"minimum at k = {best_k_on_test}")
ax.set_xlabel("k (number of neighbours)")
ax.set_ylabel("error rate")
ax.set_title("Error rate against k — measured on the test set")
ax.legend()
plt.show()

### 10.4 Why that number is not a result

The value `k` was chosen by looking at the test set, and then the test
set was used to report the score. The test set has been used twice, and
the second use is no longer an estimate of performance on unseen data —
it is the maximum of thirty noisy measurements, which is biased upward
by construction.

Do it properly: choose `k` by cross-validation **inside the training
set**, then touch the test set once.

In [ ]:
cv_error_rates = []
for k in k_values:
    pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    scores = cross_val_score(pipeline, X_knn_tr, y_knn_tr, cv=knn_cv, scoring="accuracy")
    cv_error_rates.append(1 - scores.mean())

cv_error_rates = np.array(cv_error_rates)
best_k_by_cv = int(np.argmin(cv_error_rates)) + 1

final_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k_by_cv))
final_model.fit(X_knn_tr, y_knn_tr)
honest_test_accuracy = final_model.score(X_knn_te, y_knn_te)

print(f"k chosen on the TEST set : {best_k_on_test:2}   reported accuracy {1 - test_error_rates.min():.4f}  <- optimistic")
print(f"k chosen by CV on TRAIN  : {best_k_by_cv:2}   reported accuracy {honest_test_accuracy:.4f}  <- honest")
print()
print(f"the optimism is {(1 - test_error_rates.min()) - honest_test_accuracy:+.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(list(k_values), 1 - test_error_rates, marker="o", alpha=0.8,
        label="accuracy on the test set (selection here is cheating)")
ax.plot(list(k_values), 1 - cv_error_rates, marker="s", alpha=0.8,
        label="cross-validated accuracy on the training set (correct)")
ax.axvline(best_k_on_test, color="tab:blue", linestyle=":", alpha=0.7)
ax.axvline(best_k_by_cv, color="tab:orange", linestyle=":", alpha=0.7)
ax.scatter([best_k_by_cv], [honest_test_accuracy], marker="*", s=260, color="black",
           zorder=5, label="the one honest number: chosen by CV, measured once on test")
ax.set_xlabel("k (number of neighbours)")
ax.set_ylabel("accuracy")
ax.set_title("Choosing k: on the test set, and properly")
ax.legend(fontsize=8, loc="lower right")
plt.show()

The blue curve's maximum is the number people report. The black star is
the number that means something. The difference between them is not a
bug in the code — every line ran correctly — it is a flaw in the
**procedure**, and no amount of care with the implementation detects it.

Three ways the same error appears in practice, all of them common:

1. Sweeping a hyperparameter against the test set, as here.
2. Trying several models, reporting the best test score, and not saying
   how many were tried.
3. Preprocessing — scaling, feature selection, imputation — fitted before
   the split (§6.3).

**The test set is a measurement instrument with one use.** Once you have
looked at it to make a decision, it has become part of the training
procedure.

In [ ]:
print("A small k memorises; a large k over-smooths. Both ends are visible above:")
for k in (1, best_k_by_cv, 30):
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_knn_tr, y_knn_tr)
    train_accuracy = model.score(X_knn_tr, y_knn_tr)
    cv_accuracy = cross_val_score(model, X_knn_tr, y_knn_tr, cv=knn_cv).mean()
    note = "  <- memorises the training set" if k == 1 else ""
    print(f"  k = {k:2}   train {train_accuracy:.4f}   CV {cv_accuracy:.4f}{note}")

At `k = 1` every training point is its own nearest neighbour, so training
accuracy is 1.0 and means nothing — the same empty perfection as the
unpruned tree in §8.1.

## 11. Support vector machines

### 11.1 The margin, before the API

A linear classifier draws a boundary. Many boundaries separate the same
two classes; a support vector machine picks a specific one — **the
boundary with the widest margin**, where the margin is the empty corridor
between the boundary and the nearest points of either class.

The points that sit on the edge of that corridor are the **support
vectors**. They alone determine the boundary: move any other point, or
delete it, and nothing changes.

In [ ]:
margin_rng = np.random.default_rng(SEED)
group_a = margin_rng.normal([2.0, 2.0], 0.55, size=(30, 2))
group_b = margin_rng.normal([5.0, 5.0], 0.55, size=(30, 2))
X_margin = np.vstack([group_a, group_b])
y_margin = np.hstack([np.zeros(30, dtype=int), np.ones(30, dtype=int)])

margin_svm = SVC(kernel="linear", C=1000.0, random_state=SEED).fit(X_margin, y_margin)

print("support vectors:", margin_svm.n_support_, "out of", len(X_margin), "points")
print("weight vector w:", margin_svm.coef_[0].round(4))
print("intercept b    :", round(float(margin_svm.intercept_[0]), 4))
print("margin width   :", round(2 / np.linalg.norm(margin_svm.coef_[0]), 4))

In [ ]:
def draw_boundary(ax, model, X, y, title, resolution=200):
    """Contour the decision function at -1, 0 and +1: the two margins and
    the boundary between them."""
    x_min, x_max = X[:, 0].min() - 0.6, X[:, 0].max() + 0.6
    y_min, y_max = X[:, 1].min() - 0.6, X[:, 1].max() + 0.6
    grid_x, grid_y = np.meshgrid(np.linspace(x_min, x_max, resolution),
                                 np.linspace(y_min, y_max, resolution))
    decision = model.decision_function(np.c_[grid_x.ravel(), grid_y.ravel()]).reshape(grid_x.shape)
    contours = ax.contour(grid_x, grid_y, decision, levels=[-1, 0, 1],
                          linestyles=["--", "-", "--"], colors="black", linewidths=[1, 2, 1])
    ax.clabel(contours, inline=True, fontsize=8)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=28, alpha=0.85, edgecolor="none")
    ax.set_title(title)
    return contours

fig, ax = plt.subplots(figsize=(7, 5.5))
draw_boundary(ax, margin_svm, X_margin, y_margin, "Maximum-margin boundary and its two margins")
support = margin_svm.support_vectors_
ax.scatter(support[:, 0], support[:, 1], s=190, facecolors="none",
           edgecolors="black", linewidths=1.6, label="support vectors")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.legend()
plt.show()

The solid line is the boundary, where the decision function is 0. The two
dashed lines are at -1 and +1: the edges of the margin. The circled
points are the support vectors.

Confirm the claim that only they matter, by deleting everything else:

In [ ]:
support_only_X = margin_svm.support_vectors_
support_only_y = y_margin[margin_svm.support_]
refitted = SVC(kernel="linear", C=1000.0, random_state=SEED).fit(support_only_X, support_only_y)

print(f"fitted on all {len(X_margin)} points   : w = {margin_svm.coef_[0].round(4)},"
      f" b = {margin_svm.intercept_[0]:.4f}")
print(f"fitted on {len(support_only_X)} support vectors: w = {refitted.coef_[0].round(4)},"
      f" b = {refitted.intercept_[0]:.4f}")
print("\nsame boundary:", np.allclose(margin_svm.coef_, refitted.coef_, atol=1e-3)
      and np.allclose(margin_svm.intercept_, refitted.intercept_, atol=1e-3))

### 11.2 `C`: the margin against the violations

Real classes overlap, so a corridor with nothing in it may not exist.
`C` sets the price of letting a point sit inside the margin or on the
wrong side of it.

- **Small `C`** — violations are cheap. A wide margin, more points inside
  it, a smoother boundary that ignores individual points. More bias.
- **Large `C`** — violations are expensive. A narrow margin that bends to
  classify training points correctly. More variance.

`C` is a regularisation parameter, and it trades exactly the way §7.3's
polynomial degree did.

In [ ]:
overlap_rng = np.random.default_rng(SEED)
X_overlap = np.vstack([overlap_rng.normal([2.0, 2.0], 1.15, size=(60, 2)),
                       overlap_rng.normal([4.2, 4.2], 1.15, size=(60, 2))])
y_overlap = np.hstack([np.zeros(60, dtype=int), np.ones(60, dtype=int)])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, c_value in zip(axes, [0.01, 1.0, 100.0]):
    model = SVC(kernel="linear", C=c_value, random_state=SEED).fit(X_overlap, y_overlap)
    draw_boundary(ax, model, X_overlap, y_overlap,
                  f"C = {c_value}\n{int(model.n_support_.sum())} support vectors, "
                  f"margin width {2 / np.linalg.norm(model.coef_[0]):.2f}")
    ax.set_xlabel("feature 1")
axes[0].set_ylabel("feature 2")
fig.suptitle("C controls the width of the margin and how many points are allowed inside it")
fig.tight_layout()
plt.show()

In [ ]:
svm_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
for c_value in (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0):
    model = SVC(kernel="linear", C=c_value, random_state=SEED).fit(X_overlap, y_overlap)
    scores = cross_val_score(SVC(kernel="linear", C=c_value, random_state=SEED),
                             X_overlap, y_overlap, cv=svm_cv)
    print(f"C = {c_value:7}   support vectors {int(model.n_support_.sum()):3}"
          f"   margin {2 / np.linalg.norm(model.coef_[0]):6.3f}   CV accuracy {scores.mean():.4f}")

### 11.3 Support vector regression, and recovering the boundary in
original units

The draft that this replaces broke off mid-comment here. It is the
clearest demonstration in the course that **a model fitted on scaled data
has its parameters in scaled space**, and that reporting them requires
undoing the transformation.

Start with the two noisy lines the draft built.

In [ ]:
from sklearn.svm import SVR

svr_rng = np.random.default_rng(SEED)
n_lines = 300

a_1, b_1 = 1.0, 3.0
a_2, b_2 = 2.0, 5.0

x_lines = svr_rng.uniform(0, 5, n_lines)
y_line_1 = a_1 * x_lines + b_1 + svr_rng.normal(0, 1, n_lines)
y_line_2 = a_2 * x_lines + b_2 + svr_rng.normal(0, 1, n_lines)

X_lines = x_lines.reshape(-1, 1)
x_fit = np.linspace(0, 5, 400).reshape(-1, 1)

svr_1 = make_pipeline(StandardScaler(), SVR(kernel="linear", C=100.0, epsilon=0.1)).fit(X_lines, y_line_1)
svr_2 = make_pipeline(StandardScaler(), SVR(kernel="linear", C=100.0, epsilon=0.1)).fit(X_lines, y_line_2)

fig, ax = plt.subplots()
ax.scatter(x_lines, y_line_1, s=14, alpha=0.5, label="class 0 data")
ax.plot(x_fit, svr_1.predict(x_fit), linewidth=2.5, label="SVR fit, class 0")
ax.scatter(x_lines, y_line_2, s=14, alpha=0.5, marker="d", label="class 1 data")
ax.plot(x_fit, svr_2.predict(x_fit), linewidth=2.5, label="SVR fit, class 1")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Support vector regression on two noisy lines")
ax.legend(fontsize=8)
plt.show()

`SVR` fits a tube of half-width `epsilon` around the data and penalises
only the points outside it. `epsilon=0.1` is the tolerance; `C=100`
makes excursions beyond it expensive.

Now stack the two lines into a single two-dimensional **classification**
problem and find the boundary between them.

In [ ]:
X_two_class = np.vstack([
    np.column_stack([x_lines, y_line_1]),      # class 0
    np.column_stack([x_lines, y_line_2]),      # class 1
])
y_two_class = np.hstack([np.zeros(n_lines, dtype=int), np.ones(n_lines, dtype=int)])

boundary_model = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear", C=1.0, random_state=SEED),
)
boundary_model.fit(X_two_class, y_two_class)

print("X", X_two_class.shape, " y", y_two_class.shape)
print("training accuracy:", round(boundary_model.score(X_two_class, y_two_class), 4))

### 11.4 Undoing the scaling, algebraically

The `SVC` inside that pipeline never saw the original coordinates. Its
`coef_` and `intercept_` describe a line in **scaled** space, and quoting
them as though they were in the original units would be wrong.

Write the algebra out. The scaler maps each coordinate as

$$z_1 = \\frac{x - \\mu_x}{s_x}, \\qquad z_2 = \\frac{y - \\mu_y}{s_y}$$

and the SVM's boundary in scaled space is $w_1 z_1 + w_2 z_2 + b = 0$.
Substituting and solving for $y$:

$$w_1\\frac{x - \\mu_x}{s_x} + w_2\\frac{y - \\mu_y}{s_y} + b = 0$$

$$y = \\mu_y - \\frac{s_y}{w_2}\\left(w_1\\frac{x - \\mu_x}{s_x} + b\\right)$$

which is a straight line in the original units, with

$$\\text{slope} = -\\frac{s_y w_1}{w_2 s_x}, \\qquad
\\text{intercept} = \\mu_y - \\frac{s_y}{w_2}\\left(b - \\frac{w_1 \\mu_x}{s_x}\\right)$$

In [ ]:
scaler_step = boundary_model.named_steps["standardscaler"]
svc_step = boundary_model.named_steps["svc"]

w = svc_step.coef_[0]
b = svc_step.intercept_[0]
mu_x, mu_y = scaler_step.mean_
s_x, s_y = scaler_step.scale_

print("in SCALED space:")
print(f"  w = {w.round(4)},  b = {b:.4f}")
print(f"  boundary: {w[0]:.4f} * z1 + {w[1]:.4f} * z2 + {b:.4f} = 0")
print()
print("the scaler's parameters:")
print(f"  mu = ({mu_x:.4f}, {mu_y:.4f}),  s = ({s_x:.4f}, {s_y:.4f})")

if abs(w[1]) < 1e-12:
    raise ValueError("boundary is vertical in scaled space; the y-solved form does not apply")

slope = -(s_y / w[1]) * (w[0] / s_x)
intercept = mu_y - (s_y / w[1]) * (b - w[0] * mu_x / s_x)

print()
print("in ORIGINAL units:")
print(f"  y = {slope:.4f} * x + {intercept:.4f}")

In [ ]:
# Verify the algebra numerically rather than trusting it: points ON the
# derived line must sit exactly on the pipeline's decision boundary.
check_x = np.linspace(0.5, 4.5, 7)
check_y = slope * check_x + intercept
decision_values = boundary_model.decision_function(np.column_stack([check_x, check_y]))

print("decision function at seven points on the derived line:")
print(" ", decision_values.round(10))
print("\nall essentially zero:", np.allclose(decision_values, 0, atol=1e-9))

The derived line is the decision boundary, confirmed numerically rather
than asserted. This is the check that the draft never got to.

In [ ]:
x_boundary = np.linspace(0, 5, 200)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(x_lines, y_line_1, s=14, alpha=0.45, label="class 0")
ax.scatter(x_lines, y_line_2, s=14, alpha=0.45, marker="d", label="class 1")
ax.plot(x_boundary, a_1 * x_boundary + b_1, linewidth=2, linestyle="-",
        color="tab:blue", label=f"true class 0 line: y = {a_1:.0f}x + {b_1:.0f}")
ax.plot(x_boundary, a_2 * x_boundary + b_2, linewidth=2, linestyle="-",
        color="tab:orange", label=f"true class 1 line: y = {a_2:.0f}x + {b_2:.0f}")
ax.plot(x_boundary, slope * x_boundary + intercept, "k--", linewidth=3,
        label=f"SVM boundary: y = {slope:.3f}x + {intercept:.3f}")
ax.set_xlim(0, 5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("The separating line, recovered in the original coordinates")
ax.legend(fontsize=8, loc="upper left")
plt.show()

print(f"the two true lines have slopes {a_1:.1f} and {a_2:.1f}; "
      f"the boundary's slope is {slope:.3f}, between them as it should be")

**The general lesson, which applies far beyond SVMs:** any model fitted
inside a pipeline reports its parameters in the space the last
transformer produced. Coefficients from a model trained on standardised
features are "per standard deviation", not "per unit", and comparing them
to unstandardised coefficients — or quoting them to a domain expert as
though they were in physical units — is a category error.

## 12. Kernels

### 12.1 Six datasets, generated by visible code

To see what a kernel buys you, you need data a straight line cannot
separate. These six generators build it. They are written out here, in
the notebook, for two reasons: Colab has no checkout of this repository
so an import would fail, and — more importantly — **the generators are
the teaching content**. Reading `make_xor` tells you exactly why XOR is
not linearly separable, which watching a library return two clouds does
not.

`make_moons` and `make_circles` are written by hand in the same style as
the other four, for the same reason.

In [ ]:
def make_moons_by_hand(n, rng, noise=0.25):
    """Two interleaving half-circles. Not linearly separable: any straight
    line that captures one crescent must cut through the other."""
    half = n // 2
    angle = np.linspace(0, np.pi, half)
    upper = np.column_stack([np.cos(angle), np.sin(angle)])
    lower = np.column_stack([1 - np.cos(angle), 0.5 - np.sin(angle)])
    X = np.vstack([upper, lower]) + rng.normal(0, noise, size=(2 * half, 2))
    y = np.hstack([np.zeros(half, dtype=int), np.ones(half, dtype=int)])
    return X, y

def make_circles_by_hand(n, rng, noise=0.1, factor=0.5):
    """One ring inside another. The classes differ only in RADIUS, so no
    linear function of (x, y) can separate them."""
    half = n // 2
    angle = rng.uniform(0, 2 * np.pi, half)
    outer = np.column_stack([np.cos(angle), np.sin(angle)])
    angle_inner = rng.uniform(0, 2 * np.pi, half)
    inner = factor * np.column_stack([np.cos(angle_inner), np.sin(angle_inner)])
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(2 * half, 2))
    y = np.hstack([np.zeros(half, dtype=int), np.ones(half, dtype=int)])
    return X, y

def make_xor(n, rng, noise=0.15):
    """Class is the SIGN OF THE PRODUCT of the coordinates. Each variable
    alone is useless; only their interaction carries the label."""
    X = rng.uniform(-1, 1, size=(n, 2))
    y = (X[:, 0] * X[:, 1] > 0).astype(int)
    X = X + rng.normal(0, noise, size=X.shape)
    return X, y

def make_spirals(n, rng, noise=0.08, turns=3):
    """Two spirals offset by half a turn. The boundary wraps around the
    origin, so it cannot be written as a bounded-degree polynomial."""
    half = n // 2
    theta = np.linspace(0, turns * 2 * np.pi, half)
    radius = np.linspace(0.0, 1.0, half)
    first = np.column_stack([radius * np.cos(theta), radius * np.sin(theta)])
    second = np.column_stack([radius * np.cos(theta + np.pi), radius * np.sin(theta + np.pi)])
    X = np.vstack([first, second]) + rng.normal(0, noise, size=(2 * half, 2))
    y = np.hstack([np.zeros(half, dtype=int), np.ones(half, dtype=int)])
    return X, y

def make_rings(n, rng, noise=0.05):
    """Like circles, but both classes are thin annuli at different radii."""
    half = n // 2
    angle_inner = rng.uniform(0, 2 * np.pi, half)
    angle_outer = rng.uniform(0, 2 * np.pi, half)
    radius_inner = rng.normal(0.5, noise, half)
    radius_outer = rng.normal(1.0, noise, half)
    inner = np.column_stack([radius_inner * np.cos(angle_inner), radius_inner * np.sin(angle_inner)])
    outer = np.column_stack([radius_outer * np.cos(angle_outer), radius_outer * np.sin(angle_outer)])
    X = np.vstack([inner, outer])
    y = np.hstack([np.zeros(half, dtype=int), np.ones(half, dtype=int)])
    return X, y

def make_checker(n, rng, noise=0.03, tiles=6):
    """A checkerboard. The boundary has many disconnected pieces, so the
    model needs locality rather than a single global rule."""
    side = int(np.sqrt(n))
    axis = np.linspace(-1, 1, side)
    grid_x, grid_y = np.meshgrid(axis, axis)
    X = np.column_stack([grid_x.ravel(), grid_y.ravel()])
    y = ((np.floor((grid_x + 1) * tiles / 2) + np.floor((grid_y + 1) * tiles / 2)) % 2 > 0)
    y = y.astype(int).ravel()
    X = X + rng.normal(0, noise, size=X.shape)
    chosen = rng.choice(len(X), size=n, replace=True)
    return X[chosen], y[chosen]

def load_dataset(kind, n, rng):
    """Dispatch to one of the six generators above."""
    generators = {
        "moons": make_moons_by_hand,
        "circles": make_circles_by_hand,
        "xor": make_xor,
        "spirals": make_spirals,
        "rings": make_rings,
        "checker": make_checker,
    }
    if kind not in generators:
        raise ValueError(f"unknown dataset {kind!r}; expected one of {sorted(generators)}")
    return generators[kind](n, rng)

DATASET_NAMES = ["moons", "circles", "xor", "spirals", "rings", "checker"]
print("generators defined:", DATASET_NAMES)

In [ ]:
N_POINTS = 300

fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
for ax, name in zip(axes.ravel(), DATASET_NAMES):
    X_demo_set, y_demo_set = load_dataset(name, N_POINTS, np.random.default_rng(SEED))
    ax.scatter(X_demo_set[:, 0], X_demo_set[:, 1], c=y_demo_set, cmap="coolwarm",
               s=14, alpha=0.8, edgecolor="none")
    ax.set_title(name)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    ax.set_aspect("equal")
fig.suptitle("Six datasets, none of them linearly separable")
fig.tight_layout()
plt.show()

### 12.2 The kernel trick

A linear SVM can only draw a hyperplane. The trick is to map the data
into a higher-dimensional space where a hyperplane *is* enough, and to do
it without ever computing the mapping.

The SVM's optimisation uses the training points only through **inner
products** $\\langle x_i, x_j \\rangle$. Replace every inner product with
a kernel function $K(x_i, x_j)$ and you get the same algorithm operating
in whatever space that kernel corresponds to — at the cost of evaluating
$K$, not at the cost of the higher dimension.

Make it concrete on the circles data, where the classes differ only in
radius. Add $x_1^2 + x_2^2$ as a third coordinate and the two classes
separate with a flat plane:

In [ ]:
X_circles, y_circles = load_dataset("circles", 300, np.random.default_rng(SEED))
radius_squared = X_circles[:, 0] ** 2 + X_circles[:, 1] ** 2

fig = plt.figure(figsize=(12, 4.6))

ax_flat = fig.add_subplot(1, 2, 1)
ax_flat.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap="coolwarm", s=16)
ax_flat.set_xlabel("x1")
ax_flat.set_ylabel("x2")
ax_flat.set_title("Original 2-D: no line separates them")
ax_flat.set_aspect("equal")

ax_lifted = fig.add_subplot(1, 2, 2, projection="3d")
ax_lifted.scatter(X_circles[:, 0], X_circles[:, 1], radius_squared, c=y_circles, cmap="coolwarm", s=14)
ax_lifted.set_xlabel("x1")
ax_lifted.set_ylabel("x2")
ax_lifted.set_zlabel("x1^2 + x2^2")
ax_lifted.set_title("Lifted to 3-D: a flat plane separates them")

fig.tight_layout()
plt.show()

print("a horizontal plane at x1^2 + x2^2 = 0.55 separates the classes:")
print("  accuracy of that single rule:",
      round(float(((radius_squared > 0.55).astype(int) == y_circles).mean()), 4))

That third coordinate is what a polynomial kernel of degree 2 provides
implicitly. The RBF kernel,

$$K(x_i, x_j) = \\exp\\left(-\\gamma \\lVert x_i - x_j \\rVert^2\\right)$$

corresponds to a space of infinite dimension, which is why it can fit
essentially any boundary — and why it needs to be restrained.

**`gamma` is the reach of one training point.** Large `gamma` makes the
exponential decay fast, so each point influences only its immediate
neighbourhood and the boundary becomes a collection of small islands.
Small `gamma` gives each point a long reach and a smooth boundary.

In [ ]:
distances = np.linspace(0, 3, 300)

fig, ax = plt.subplots(figsize=(7, 4))
for gamma_value in (0.1, 0.5, 2.0, 10.0):
    ax.plot(distances, np.exp(-gamma_value * distances ** 2), label=f"gamma = {gamma_value}")
ax.set_xlabel("distance between two points")
ax.set_ylabel("kernel value (their similarity)")
ax.set_title("gamma sets how quickly similarity falls off with distance")
ax.legend()
plt.show()

print("at gamma = 10, two points 1.0 apart have similarity",
      round(float(np.exp(-10 * 1.0 ** 2)), 6), "— effectively unrelated")
print("at gamma = 0.1, the same pair has similarity",
      round(float(np.exp(-0.1 * 1.0 ** 2)), 6), "— still strongly related")

### 12.3 Six kernels against six datasets

The comparison the draft set up but never ran. Each panel contours the
decision function at -1, 0 and +1, so you see the boundary **and both
margins**.

In [ ]:
kernel_models = [
    ("Linear", make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0, random_state=SEED))),
    ("Poly deg=3", make_pipeline(StandardScaler(), SVC(kernel="poly", degree=3, C=5.0,
                                                      coef0=1.0, gamma="scale", random_state=SEED))),
    ("RBF gamma=scale", make_pipeline(StandardScaler(), SVC(kernel="rbf", C=5.0,
                                                           gamma="scale", random_state=SEED))),
    ("RBF gamma=auto", make_pipeline(StandardScaler(), SVC(kernel="rbf", C=5.0,
                                                          gamma="auto", random_state=SEED))),
    ("RBF smooth (C=1, g=0.5)", make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0,
                                                                   gamma=0.5, random_state=SEED))),
    ("RBF wiggly (C=50, g=5)", make_pipeline(StandardScaler(), SVC(kernel="rbf", C=50.0,
                                                                  gamma=5.0, random_state=SEED))),
]

# RUNTIME NOTE. The source material used a 400x400 mesh, which is 160,000
# decision_function evaluations per panel and 5.76 million over the whole
# 6x6 grid. At 150x150 it is 22,500 per panel and about 810,000 in total --
# roughly a seventh of the work, for a boundary that is visually identical
# at this figure size. Raise MESH_RESOLUTION to 400 if you want the
# original; expect the six figures below to take several minutes.
MESH_RESOLUTION = 150

print(f"{len(kernel_models)} models x {len(DATASET_NAMES)} datasets = "
      f"{len(kernel_models) * len(DATASET_NAMES)} fits, mesh {MESH_RESOLUTION}x{MESH_RESOLUTION}")

In [ ]:
kernel_accuracy = {}
grid_start = time.time()

for name in DATASET_NAMES:
    X_set, y_set = load_dataset(name, N_POINTS, np.random.default_rng(SEED))

    fig, axes = plt.subplots(2, 3, figsize=(13, 7.4))
    for ax, (model_name, model) in zip(axes.ravel(), kernel_models):
        model.fit(X_set, y_set)
        accuracy = model.score(X_set, y_set)
        kernel_accuracy[(name, model_name)] = accuracy
        draw_boundary(ax, model, X_set, y_set,
                      f"{model_name}\ntraining accuracy {accuracy:.3f}",
                      resolution=MESH_RESOLUTION)
        ax.set_xlabel("feature 1")
        ax.set_ylabel("feature 2")

    fig.suptitle(f"Kernel SVMs on the '{name}' dataset", fontsize=13)
    fig.tight_layout()
    plt.show()

print(f"grid completed in {time.time() - grid_start:.1f} s")

In [ ]:
accuracy_table = pd.DataFrame(
    [[kernel_accuracy[(d, m)] for m, _ in kernel_models] for d in DATASET_NAMES],
    index=DATASET_NAMES, columns=[m for m, _ in kernel_models],
)
print("TRAINING accuracy (not a performance estimate — see the note below)")
print(accuracy_table.round(3).to_string())

Read the table and the figures together.

**Linear fails on all six**, as designed — they were built to defeat it.
On spirals and checker it is at or below the coin-flip baseline of §9.4.

**Poly degree 3** solves circles and rings outright, does well on moons
and XOR, and cannot represent the checkerboard or the spirals: a
degree-3 surface has nowhere near enough pieces.

**RBF at its default settings solves four of the six.** It matches the
polynomial on circles, rings, moons and XOR, and it does **not** handle
spirals or the checkerboard — both sit around 0.6 on training data. That
is worth dwelling on, because "RBF can fit anything" is a thing people
say. It can, in principle; at `gamma="scale"` on this data it does not,
because the default reach is too long for boundaries this finely divided.

**`gamma=scale` against `gamma=auto`.** `scale` uses
`1 / (n_features × X.var())` and `auto` uses `1 / n_features`. Every
model here is fitted inside a pipeline whose scaler makes the variance 1,
so the two are identical in this table — the columns agree to four
decimal places. On unscaled data they differ substantially.

**The last two panels are the lesson.** `C=1, gamma=0.5` gives a smooth
boundary. `C=50, gamma=5` wraps individual points in small islands and
scores higher on **training** data everywhere — it is the degree-15
polynomial of §7.3 in a different costume.

Every number in that table is a training accuracy, so the wiggly model
looks best everywhere. Score them honestly:

In [ ]:
honest_rows = []
for name in DATASET_NAMES:
    X_set, y_set = load_dataset(name, N_POINTS, np.random.default_rng(SEED))
    row = {}
    for model_name, model in kernel_models:
        scores = cross_val_score(model, X_set, y_set,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
        row[model_name] = scores.mean()
    honest_rows.append(row)

honest_table = pd.DataFrame(honest_rows, index=DATASET_NAMES)
print("CROSS-VALIDATED accuracy")
print(honest_table.round(3).to_string())
print()
print("training minus cross-validated, for the two RBF settings:")
gap = (accuracy_table[["RBF smooth (C=1, g=0.5)", "RBF wiggly (C=50, g=5)"]]
       - honest_table[["RBF smooth (C=1, g=0.5)", "RBF wiggly (C=50, g=5)"]])
print(gap.round(3).to_string())

Now the picture splits, and the split is the useful part.

On **moons** and **XOR** the wiggly model's training advantage reverses
under cross-validation: it scored higher on the data it was shown and
lower on data it was not. That is overfitting, plainly.

On **spirals** and **checker** the wiggly model is genuinely better, by a
wide margin on checker. Those boundaries really are finely divided, so
the short reach that was excessive on moons is necessary here.

**High `gamma` is not "wrong". It is a claim that the boundary is local,
and whether that claim is true depends on the data.** The train-minus-
validation gap is what tells you which case you are in: the wiggly model
has the larger gap on every dataset, but on checker it is large *and*
still the better model, because it started from so much further ahead.

Read the gap as a warning to check, not as a verdict. The verdict is the
cross-validated score.

This is also why §6.4 tuned `C` and `gamma` by grid search rather than
picking defaults: there is no setting that is right for all six of these,
and there was no way to know which you had without measuring.

## 13. Clustering

Everything so far was **supervised**: the data came with labels and the
question was how well a model reproduced them. Clustering has no labels.
That changes the epistemics completely, and this section is as much about
that as about the algorithms.

### 13.1 There is no ground truth, and comparing against labels is a
teaching device

Every worked example below compares the clusters found against known
labels. **That is not how clustering is used.** If you had the labels you
would fit a classifier. The labels are here so that you can see what the
algorithm did; in the situation where you would actually reach for
clustering, they do not exist.

The distinction to keep:

- **Internal validation** — inertia, silhouette, BIC. Computed from the
  data and the clustering alone. Available in real use.
- **External validation** — adjusted Rand index, homogeneity. Compares a
  clustering against known labels. Available only in a teaching example
  or an evaluation study.

### 13.2 k-means on blobs built by hand

`make_blobs` is banned in this course (Lesson 2, and
`practice/README.md` §6), and it is no loss: a blob is a draw from a
multivariate normal, which Practice 1 §4.9 already built.

In [ ]:
from sklearn.cluster import KMeans

blob_rng = np.random.default_rng(SEED)
blob_centres = np.array([[-4.0, -2.0], [0.0, 3.5], [4.0, -1.0], [6.5, 4.0]])
points_per_blob = 75

X_blobs4 = np.vstack([
    blob_rng.multivariate_normal(centre, np.eye(2), points_per_blob)
    for centre in blob_centres
])
y_blobs4 = np.repeat(np.arange(len(blob_centres)), points_per_blob)

print("X", X_blobs4.shape, " true groups", np.bincount(y_blobs4))
print("centres used to generate the data:")
print(blob_centres)

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(X_blobs4)

print("inertia (within-cluster sum of squares):", round(kmeans.inertia_, 4))
print("\ncluster centres found:")
print(np.round(kmeans.cluster_centers_[np.argsort(kmeans.cluster_centers_[:, 0])], 4))
print("\ntrue centres, sorted the same way:")
print(blob_centres[np.argsort(blob_centres[:, 0])])

The centres are recovered to within about a tenth of a unit. Note that
the **cluster numbers are arbitrary** — k-means has no way to know that
its cluster 2 is your group 0. Any comparison against labels has to be
invariant to relabelling, which is what the adjusted Rand index in §13.5
is for.

### 13.3 Choosing k: the elbow and the silhouette

`n_clusters` must be chosen in advance, and the data does not announce
the answer.

**Inertia** is the sum of squared distances from each point to its
cluster centre. It falls monotonically as `k` rises — at `k = n` every
point is its own cluster and inertia is zero — so you cannot minimise it.
You look for the **elbow**, where the improvement stops being worth it.

In [ ]:
k_range = range(1, 12)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_blobs4).inertia_
            for k in k_range]

from sklearn.metrics import silhouette_score

silhouettes = {}
for k in range(2, 7):                       # silhouette is undefined for k = 1
    labels_k = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X_blobs4)
    silhouettes[k] = silhouette_score(X_blobs4, labels_k)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].axvline(4, color="tab:red", linestyle=":", label="true number of groups")
axes[0].set_xlabel("k")
axes[0].set_ylabel("inertia")
axes[0].set_title("Elbow: inertia always falls, so look for the bend")
axes[0].legend()

axes[1].plot(list(silhouettes), list(silhouettes.values()), marker="s", color="tab:orange")
axes[1].axvline(4, color="tab:red", linestyle=":", label="true number of groups")
axes[1].set_xlabel("k")
axes[1].set_ylabel("mean silhouette")
axes[1].set_title("Silhouette: higher is better, so this one has a maximum")
axes[1].legend()

fig.tight_layout()
plt.show()

for k, score in silhouettes.items():
    marker = "  <- best" if score == max(silhouettes.values()) else ""
    print(f"k = {k}   silhouette {score:.4f}{marker}")

**Silhouette** measures, for each point, how much closer it is to its own
cluster than to the nearest other cluster, on a scale from -1 to +1. It
has a maximum, so unlike inertia it can be optimised directly.

Both agree on `k = 4` here, which is the honest answer because the data
really does have four well-separated round groups. §14 shows both being
wrong.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
for ax, k in zip(axes.ravel(), [2, 3, 4, 5]):
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_blobs4)
    ax.scatter(X_blobs4[:, 0], X_blobs4[:, 1], c=model.labels_, cmap="tab10", s=16, alpha=0.75)
    ax.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1],
               marker="X", s=220, c="black", label="centres")
    ax.set_title(f"k = {k}   inertia {model.inertia_:.0f}   silhouette {silhouettes[k]:.3f}")
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    ax.legend(fontsize=8)
fig.suptitle("The same data forced into 2, 3, 4 and 5 clusters")
fig.tight_layout()
plt.show()

k-means returns `k` clusters whatever `k` you pass. At `k = 2` it merges
two real groups; at `k = 5` it splits one. **Neither looks like a
failure from the inside** — the algorithm reports success in all four
cases. The only signal is the internal measure, and the only defence is
to look at the picture.

### 13.4 `n_init`, and the initialisation problem

k-means minimises inertia by alternating two steps from a random start.
It converges to a **local** minimum, and different starts give different
answers. `n_init` is how many starts it tries before keeping the best.

In [ ]:
print("A single start, from ten different seeds:")
single_start = []
for seed in range(10):
    model = KMeans(n_clusters=4, n_init=1, init="random", random_state=seed).fit(X_blobs4)
    single_start.append(model.inertia_)
    
single_start = np.array(single_start)
print(" ", single_start.round(1))
print(f"  best {single_start.min():.1f}   worst {single_start.max():.1f}"
      f"   spread {single_start.max() - single_start.min():.1f}")

print("\nTen starts, keeping the best (the default), from the same ten seeds:")
ten_starts = np.array([
    KMeans(n_clusters=4, n_init=10, init="random", random_state=seed).fit(X_blobs4).inertia_
    for seed in range(10)
])
print(" ", ten_starts.round(1))
print(f"  best {ten_starts.min():.1f}   worst {ten_starts.max():.1f}"
      f"   spread {ten_starts.max() - ten_starts.min():.1f}")

A single random start lands in a bad local minimum some of the time.
Averaging over ten starts removes almost all of that variation.

`init="k-means++"` — the default — is smarter still: it spreads the
initial centres out rather than placing them uniformly at random, which
makes a bad start much less likely.

In [ ]:
for init_method in ("random", "k-means++"):
    results = np.array([
        KMeans(n_clusters=4, n_init=1, init=init_method, random_state=seed).fit(X_blobs4).inertia_
        for seed in range(20)
    ])
    print(f"{init_method:11} with n_init=1 over 20 seeds: "
          f"best {results.min():7.1f}  worst {results.max():7.1f}  "
          f"bad starts {(results > results.min() * 1.05).sum():2}/20")

**The drafts this replaces called `KMeans` with neither `random_state`
nor an explicit `n_init`**, including inside the elbow loop. Every number
they produced changed on every run, and the elbow plot could bend in a
different place each time it was drawn.

### 13.5 Clustering a high-dimensional dataset

Apply it to the digits from Practice 2 §15. Ten classes, no labels given to
the algorithm.

Cluster the **PCA reduction**, not the t-SNE projection. Practice 2 §15.7
caveat 2: distances in a t-SNE plot are not meaningful, so a
distance-based algorithm run on them measures an artifact of the
optimisation.

In [ ]:
mnist = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/mnist_sample.csv"
)
digit_labels = mnist["label"].values
pixels = mnist.drop(columns="label").values

mnist_scaled = StandardScaler().fit_transform(pixels)
mnist_pca_50 = PCA(n_components=50, random_state=SEED).fit_transform(mnist_scaled)

print("pixels", pixels.shape, "-> PCA", mnist_pca_50.shape)
print("true digit counts:", np.bincount(digit_labels))

In [ ]:
# LOCAL ALTERNATIVE. Run EITHER this cell OR the one above, not both.
# Above reads over the network and is what Colab needs. This one reads
# the same files from your checkout, for the dev-env container. The
# data is identical; only the address differs.
mnist = pd.read_csv("../datasets/mnist_sample.csv")
digit_labels = mnist["label"].values
pixels = mnist.drop(columns="label").values

mnist_scaled = StandardScaler().fit_transform(pixels)
mnist_pca_50 = PCA(n_components=50, random_state=SEED).fit_transform(mnist_scaled)

print("pixels", pixels.shape, "-> PCA", mnist_pca_50.shape)
print("true digit counts:", np.bincount(digit_labels))

In [ ]:
from sklearn.metrics import adjusted_rand_score, homogeneity_score, completeness_score

digit_kmeans = KMeans(n_clusters=10, n_init=10, random_state=SEED).fit(mnist_pca_50)

print("INTERNAL measures — computable without labels, so usable in real work:")
print(f"  inertia    {digit_kmeans.inertia_:,.0f}")
print(f"  silhouette {silhouette_score(mnist_pca_50, digit_kmeans.labels_):.4f}")
print()
print("EXTERNAL measures — need the labels, so available only here:")
print(f"  adjusted Rand index {adjusted_rand_score(digit_labels, digit_kmeans.labels_):.4f}")
print(f"  homogeneity         {homogeneity_score(digit_labels, digit_kmeans.labels_):.4f}")
print(f"  completeness        {completeness_score(digit_labels, digit_kmeans.labels_):.4f}")

The **adjusted Rand index** compares two partitions while ignoring the
labels attached to them, which is what makes it usable when cluster 3
might correspond to digit 7. It is 0 for a random partition and 1 for a
perfect match, and it is *adjusted* precisely so that chance agreement
scores 0 rather than something positive.

A silhouette near 0.06 says the clusters barely separate in this space,
and the ARI says the partition nonetheless has real agreement with the
digits. Both are true: the digit classes overlap heavily in pixel space
(Practice 2 §15.5 showed this), so the groups are genuine but not
well-isolated.

In [ ]:
crosstab = pd.crosstab(digit_labels, digit_kmeans.labels_,
                       rownames=["true digit"], colnames=["cluster"])

fig, ax = plt.subplots(figsize=(8.5, 6))
sns.heatmap(crosstab, annot=True, fmt="d", cmap="viridis", cbar_kws={"label": "images"}, ax=ax)
ax.set_title("k-means clusters against true digits (the algorithm never saw the digits)")
plt.show()

dominant = crosstab.idxmax(axis=0)
print("each cluster's most common digit:", dict(dominant))
print("distinct digits claimed:", dominant.nunique(), "of 10")

Some clusters map cleanly onto a single digit; others merge two that look
alike. If the same digit dominates more than one cluster, k-means has
split it — usually because that digit is written in two visibly different
ways.

## 14. Gaussian mixtures, and where k-means fails

### 14.1 The assumption k-means makes

k-means assigns each point to the nearest centre in Euclidean distance.
That is equivalent to assuming every cluster is **spherical, of similar
size, and of similar spread**. When the truth is elongated or tilted,
the assumption is wrong and the failure is visible.

Build exactly that case, with the non-diagonal covariances from Practice 1
§4.9.

In [ ]:
from sklearn.utils import shuffle

tilt_rng = np.random.default_rng(SEED)
cov_a = np.array([[2.0, 2.0], [2.0, 5.0]])       # tilted one way
cov_b = np.array([[7.0, -2.0], [-2.0, 3.0]])     # tilted the other, and wider

group_a = tilt_rng.multivariate_normal([0.0, 0.0], cov_a, 300)
group_b = tilt_rng.multivariate_normal([4.0, 4.0], cov_b, 300)

X_tilted = np.vstack([group_a, group_b])
y_tilted = np.hstack([np.zeros(300, dtype=int), np.ones(300, dtype=int)])
X_tilted, y_tilted = shuffle(X_tilted, y_tilted, random_state=SEED)

print("covariance A:\n", cov_a, "\n correlation", round(2 / np.sqrt(2 * 5), 4))
print("covariance B:\n", cov_b, "\n correlation", round(-2 / np.sqrt(7 * 3), 4))

In [ ]:
from sklearn.mixture import GaussianMixture

tilted_kmeans = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(X_tilted)
tilted_gmm = GaussianMixture(n_components=2, covariance_type="full",
                             random_state=SEED).fit(X_tilted)
gmm_labels = tilted_gmm.predict(X_tilted)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharex=True, sharey=True)
for ax, (labels, title) in zip(axes, [
    (y_tilted, "truth"),
    (tilted_kmeans, f"k-means (ARI {adjusted_rand_score(y_tilted, tilted_kmeans):.3f})"),
    (gmm_labels, f"GMM, full covariance (ARI {adjusted_rand_score(y_tilted, gmm_labels):.3f})"),
]):
    ax.scatter(X_tilted[:, 0], X_tilted[:, 1], c=labels, cmap="coolwarm", s=12, alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel("feature 1")
    ax.set_aspect("equal")
axes[0].set_ylabel("feature 2")
fig.suptitle("Two tilted, differently shaped clusters")
fig.tight_layout()
plt.show()

print(f"k-means ARI {adjusted_rand_score(y_tilted, tilted_kmeans):.4f}")
print(f"GMM     ARI {adjusted_rand_score(y_tilted, gmm_labels):.4f}")

The two are close — about 0.72 against 0.73. Say so rather than claiming
a dramatic win: these two clusters are far enough apart that approximating
each by a circle costs little, and k-means does an acceptable job.

What the GMM has that k-means does not is the ability to state the shape
it found. It fits a **mean and a full covariance matrix** per component,
so it recovers the generating parameters — including the tilt:

In [ ]:
order = np.argsort(tilted_gmm.means_[:, 0])
print("means recovered:")
print(np.round(tilted_gmm.means_[order], 3), "   true: [0, 0] and [4, 4]")
print("\ncovariance recovered for the first component:")
print(np.round(tilted_gmm.covariances_[order][0], 3))
print("true:\n", cov_a)
print("\ncovariance recovered for the second component:")
print(np.round(tilted_gmm.covariances_[order][1], 3))
print("true:\n", cov_b)

### 14.2 Making the assumption bite

The clusters above are separated along the same direction they are
stretched, which is the easy case. The assumption fails hardest when the
clusters are **long in one direction and separated in another**: k-means
must cut perpendicular to the direction of greatest spread, and the
groups are not separated in that direction.

Same idea, sharper geometry. Both components are strongly tilted, they
differ in size, and their centres are offset across the tilt rather than
along it.

In [ ]:
sharp_rng = np.random.default_rng(SEED)
cov_long = np.array([[9.0, 8.0], [8.0, 9.0]])      # correlation 0.89, very elongated
cov_short = np.array([[6.0, 5.0], [5.0, 6.0]])     # same tilt, smaller

X_sharp = np.vstack([
    sharp_rng.multivariate_normal([0.0, 0.0], cov_long, 300),
    sharp_rng.multivariate_normal([0.0, 7.0], cov_short, 300),   # offset ACROSS the tilt
])
y_sharp = np.hstack([np.zeros(300, dtype=int), np.ones(300, dtype=int)])
X_sharp, y_sharp = shuffle(X_sharp, y_sharp, random_state=SEED)

sharp_kmeans = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(X_sharp)
sharp_full = GaussianMixture(n_components=2, covariance_type="full",
                             random_state=SEED).fit(X_sharp).predict(X_sharp)
sharp_spherical = GaussianMixture(n_components=2, covariance_type="spherical",
                                  random_state=SEED).fit(X_sharp).predict(X_sharp)

print(f"k-means            ARI {adjusted_rand_score(y_sharp, sharp_kmeans):.4f}")
print(f"GMM, full          ARI {adjusted_rand_score(y_sharp, sharp_full):.4f}")
print(f"GMM, spherical     ARI {adjusted_rand_score(y_sharp, sharp_spherical):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.4), sharex=True, sharey=True)
panels = [
    (y_sharp, "truth"),
    (sharp_kmeans, f"k-means\nARI {adjusted_rand_score(y_sharp, sharp_kmeans):.3f}"),
    (sharp_spherical, f"GMM, spherical\nARI {adjusted_rand_score(y_sharp, sharp_spherical):.3f}"),
    (sharp_full, f"GMM, full covariance\nARI {adjusted_rand_score(y_sharp, sharp_full):.3f}"),
]
for ax, (labels, title) in zip(axes, panels):
    ax.scatter(X_sharp[:, 0], X_sharp[:, 1], c=labels, cmap="coolwarm", s=11, alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel("feature 1")
    ax.set_aspect("equal")
axes[0].set_ylabel("feature 2")
fig.suptitle("Elongated in one direction, separated in another")
fig.tight_layout()
plt.show()

Now the failure is unmistakable. k-means splits the data **left from
right**, cutting across both true clusters, because the greatest spread
is along the diagonal and that is where cutting reduces inertia most. The
true structure — an upper cluster and a lower one — is invisible to it.

Look at the third panel. **A Gaussian mixture restricted to spherical
components fails in exactly the same way**, and scores essentially the
same ARI as k-means. Switch to `covariance_type="full"` and it recovers
the structure perfectly.

That comparison is the point of this section. The failure was never about
the algorithm's name; it was about the assumption that clusters are
round. k-means makes that assumption unavoidably. A GMM makes it
optionally, through `covariance_type`.

### 14.3 What each `covariance_type` costs

In [ ]:
for covariance_type in ("full", "tied", "diag", "spherical"):
    model = GaussianMixture(n_components=2, covariance_type=covariance_type,
                            random_state=SEED).fit(X_tilted)
    labels_ct = model.predict(X_tilted)
    print(f"covariance_type={covariance_type:10} "
          f"ARI {adjusted_rand_score(y_tilted, labels_ct):.4f}   "
          f"parameters {int(model._n_parameters()):3}   BIC {model.bic(X_tilted):9.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True, sharey=True)
for ax, covariance_type in zip(axes, ["full", "spherical"]):
    model = GaussianMixture(n_components=2, covariance_type=covariance_type,
                            random_state=SEED).fit(X_tilted)
    labels_ct = model.predict(X_tilted)
    ax.scatter(X_tilted[:, 0], X_tilted[:, 1], c=labels_ct, cmap="coolwarm", s=12, alpha=0.75)
    ax.set_title(f"covariance_type='{covariance_type}'   "
                 f"ARI {adjusted_rand_score(y_tilted, labels_ct):.3f}")
    ax.set_xlabel("feature 1")
    ax.set_aspect("equal")
axes[0].set_ylabel("feature 2")
fig.suptitle("The same model class, one assumption changed")
fig.tight_layout()
plt.show()

### 14.4 Soft assignments

k-means gives each point one label. A mixture model gives a
**probability per component**, so the points in the overlap can be
identified as uncertain instead of being forced into a side.

In [ ]:
probabilities = tilted_gmm.predict_proba(X_tilted)
confidence = probabilities.max(axis=1)

print("confidence of the assignment, per point:")
print(f"  confident (>0.99): {(confidence > 0.99).sum():4} points")
print(f"  uncertain (<0.80): {(confidence < 0.80).sum():4} points")
print(f"  near a coin flip (<0.60): {(confidence < 0.60).sum():4} points")

fig, ax = plt.subplots(figsize=(7.5, 5.5))
scatter = ax.scatter(X_tilted[:, 0], X_tilted[:, 1], c=confidence,
                     cmap="viridis", s=16, vmin=0.5, vmax=1.0)
fig.colorbar(scatter, ax=ax, label="probability of the assigned component")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("Soft assignment: the uncertain points are exactly the overlap")
ax.set_aspect("equal")
plt.show()

The dark band running between the two clouds is where the model says it
does not know. That information does not exist in a k-means result, and
in an application it is often the most actionable output — those are the
cases to send to a human.

### 14.5 Choosing the number of components: BIC against silhouette

The draft chose the number of GMM components by silhouette. Silhouette
is built for compact, well-separated, roughly spherical clusters — the
very assumption this data violates.

A mixture model is a probability model, so it has a **likelihood**, and
the information criteria use it. Both **BIC** and **AIC** balance fit
against the number of parameters; **lower is better** for both, and BIC
penalises complexity more heavily.

On the two-cluster data above all three criteria agree on 2, so there is
nothing to choose between them. Make the problem slightly harder — the
situation in which the choice of criterion actually matters — by adding a
**third** component that overlaps one of the existing two. This is the
realistic case: groups that are genuinely distinct but not well
separated.

In [ ]:
three_rng = np.random.default_rng(SEED)
cov_c = np.array([[3.0, 1.5], [1.5, 2.0]])

X_three = np.vstack([
    three_rng.multivariate_normal([0.0, 0.0], cov_a, 300),
    three_rng.multivariate_normal([4.0, 4.0], cov_b, 300),
    three_rng.multivariate_normal([6.0, 6.5], cov_c, 300),    # overlaps the second
])
y_three = np.repeat([0, 1, 2], 300)
X_three, y_three = shuffle(X_three, y_three, random_state=SEED)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(X_three[:, 0], X_three[:, 1], c=y_three, cmap="coolwarm", s=12, alpha=0.75)
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("Three tilted components; the second and third overlap heavily")
ax.set_aspect("equal")
plt.show()

In [ ]:
component_range = range(1, 9)
bic_values, aic_values, silhouette_values = [], [], []

for n_components in component_range:
    model = GaussianMixture(n_components=n_components, covariance_type="full",
                            random_state=SEED).fit(X_three)
    bic_values.append(model.bic(X_three))
    aic_values.append(model.aic(X_three))
    if n_components >= 2:
        silhouette_values.append(silhouette_score(X_three, model.predict(X_three)))
    else:
        silhouette_values.append(np.nan)      # silhouette is undefined for one cluster

results = pd.DataFrame({
    "n_components": list(component_range),
    "BIC": bic_values,
    "AIC": aic_values,
    "silhouette": silhouette_values,
})
print(results.round(2).to_string(index=False))
print()
print("BIC chooses       :", int(results.loc[results['BIC'].idxmin(), 'n_components']))
print("AIC chooses       :", int(results.loc[results['AIC'].idxmin(), 'n_components']))
print("silhouette chooses:", int(results.loc[results['silhouette'].idxmax(), 'n_components']))
print("the truth is      : 3")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].plot(results["n_components"], results["BIC"], marker="o", label="BIC")
axes[0].plot(results["n_components"], results["AIC"], marker="s", label="AIC")
axes[0].axvline(3, color="tab:red", linestyle=":", label="true number of components")
axes[0].set_xlabel("number of components")
axes[0].set_ylabel("information criterion (lower is better)")
axes[0].set_title("BIC and AIC use the model's likelihood")
axes[0].legend()

axes[1].plot(results["n_components"], results["silhouette"], marker="^", color="tab:orange")
axes[1].axvline(3, color="tab:red", linestyle=":", label="true number of components")
axes[1].set_xlabel("number of components")
axes[1].set_ylabel("mean silhouette (higher is better)")
axes[1].set_title("Silhouette assumes compact spherical clusters")
axes[1].legend()

fig.tight_layout()
plt.show()

**BIC and AIC find the third component. The silhouette does not.**

The silhouette is not malfunctioning; it is answering its own question
correctly. It measures how much closer each point is to its own cluster
than to the next one, which rewards compact, well-separated groups. The
second and third components overlap, so merging them genuinely produces
a more compact two-cluster partition — and that is the answer the
silhouette reports. BIC asks a different question: which number of
components makes the observed data most likely, after paying for each
extra parameter. That question has the right answer here.

**Use the criterion that matches the model.** A mixture model has a
likelihood, so use BIC or AIC. k-means has no probability model behind
it, so it has no likelihood and the silhouette is the available option
there. Selecting the number of mixture components by silhouette, as the
draft did, asks a compactness question of a model that was never trying
to produce compact clusters, and it will systematically under-count
whenever the components overlap.

AIC penalises extra parameters less than BIC, so it tends to choose more
components. When they disagree, BIC is the more conservative choice.

## 15. One honest end-to-end run

Everything above, applied once, in order, with the test set touched
exactly once at the end. This is the shape a real piece of work has.

The task: predict survival on the Titanic.

### 15.1 Load and clean, recording every decision

In [ ]:
titanic_raw = pd.read_csv(
    "https://raw.githubusercontent.com/djnzx/ida-practice-3d/main/practice/datasets/titanic.csv"
)

decisions = []
working = titanic_raw.copy()

working["cabin_recorded"] = working["Cabin"].notna().astype(int)
working = working.drop(columns=["Cabin"])
decisions.append("Cabin: 77% missing and not missing at random (Lesson 2 section 5.3). "
                 "Dropped the value, kept the indicator.")

working = working.drop(columns=["PassengerId", "Name", "Ticket"])
decisions.append("PassengerId, Name, Ticket: identifiers with 891/891/681 distinct "
                 "values. Dropped; one-hot would exceed the row count.")

working["Age"] = working["Age"].fillna(
    working.groupby(["Pclass", "Sex"])["Age"].transform("median")
)
decisions.append("Age: 177 missing (20%), filled with the (Pclass, Sex) group median. "
                 "A constant fill would have cut the standard deviation by a tenth.")

working["Embarked"] = working["Embarked"].fillna(working["Embarked"].mode()[0])
decisions.append("Embarked: 2 missing (0.2%), filled with the mode. Defensible "
                 "because it is two rows.")

working["Sex"] = (working["Sex"] == "female").astype(int)
working = pd.get_dummies(working, columns=["Embarked"], prefix="embarked", dtype=int)
decisions.append("Sex: binary, encoded 0/1. Embarked: nominal, one-hot encoded, "
                 "not label encoded (section 10.2).")

decisions.append("Pclass: left as 1/2/3. The ordering is real (Lesson 2 section 13.3).")

y_final = working.pop("Survived").values
X_final = working.values.astype(float)
final_feature_names = list(working.columns)

print("features:", final_feature_names)
print("X", X_final.shape, " y", y_final.shape, " missing:", int(np.isnan(X_final).sum()))
print()
for i, decision in enumerate(decisions, 1):
    print(f"{i}. {decision}")

In [ ]:
# LOCAL ALTERNATIVE. Run EITHER this cell OR the one above, not both.
# Above reads over the network and is what Colab needs. This one reads
# the same files from your checkout, for the dev-env container. The
# data is identical; only the address differs.
titanic_raw = pd.read_csv("../datasets/titanic.csv")

decisions = []
working = titanic_raw.copy()

working["cabin_recorded"] = working["Cabin"].notna().astype(int)
working = working.drop(columns=["Cabin"])
decisions.append("Cabin: 77% missing and not missing at random (Lesson 2 section 5.3). "
                 "Dropped the value, kept the indicator.")

working = working.drop(columns=["PassengerId", "Name", "Ticket"])
decisions.append("PassengerId, Name, Ticket: identifiers with 891/891/681 distinct "
                 "values. Dropped; one-hot would exceed the row count.")

working["Age"] = working["Age"].fillna(
    working.groupby(["Pclass", "Sex"])["Age"].transform("median")
)
decisions.append("Age: 177 missing (20%), filled with the (Pclass, Sex) group median. "
                 "A constant fill would have cut the standard deviation by a tenth.")

working["Embarked"] = working["Embarked"].fillna(working["Embarked"].mode()[0])
decisions.append("Embarked: 2 missing (0.2%), filled with the mode. Defensible "
                 "because it is two rows.")

working["Sex"] = (working["Sex"] == "female").astype(int)
working = pd.get_dummies(working, columns=["Embarked"], prefix="embarked", dtype=int)
decisions.append("Sex: binary, encoded 0/1. Embarked: nominal, one-hot encoded, "
                 "not label encoded (section 10.2).")

decisions.append("Pclass: left as 1/2/3. The ordering is real (Lesson 2 section 13.3).")

y_final = working.pop("Survived").values
X_final = working.values.astype(float)
final_feature_names = list(working.columns)

print("features:", final_feature_names)
print("X", X_final.shape, " y", y_final.shape, " missing:", int(np.isnan(X_final).sum()))
print()
for i, decision in enumerate(decisions, 1):
    print(f"{i}. {decision}")

### 15.2 Split once, and put the test set away

Nothing below looks at `X_test` until §15.5.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=SEED, stratify=y_final
)

print(f"development set {X_dev.shape}  — everything below uses only this")
print(f"test set        {X_test.shape}  — opened once, in section 15.5")
print(f"class balance preserved: dev {y_dev.mean():.4f}, test {y_test.mean():.4f}, "
      f"full {y_final.mean():.4f}")

### 15.3 Compare candidate models by cross-validation

Every candidate is a `Pipeline`, so the scaler is re-fitted inside each
fold (§6.3). The tree does not need scaling but including it costs
nothing and keeps the comparison uniform.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

candidates = {
    "baseline (majority class)": DummyClassifier(strategy="most_frequent"),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
    "KNN (k=10)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=10)),
    "decision tree (depth 4)": make_pipeline(StandardScaler(), DecisionTreeClassifier(max_depth=4, random_state=SEED)),
    "random forest": make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=200, random_state=SEED)),
    "SVM, RBF": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED)),
}

final_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
comparison = []
for name, model in candidates.items():
    scores = cross_val_score(model, X_dev, y_dev, cv=final_cv, scoring="accuracy")
    f1_scores = cross_val_score(model, X_dev, y_dev, cv=final_cv, scoring="f1")
    comparison.append({
        "model": name,
        "cv_accuracy": scores.mean(),
        "cv_std": scores.std(),
        "cv_f1": f1_scores.mean(),
    })

comparison_table = pd.DataFrame(comparison).set_index("model")
print(comparison_table.round(4).to_string())

A random forest is included because it is the natural extension of §8:
many trees, each on a random subsample of rows and features, voting. It
is a standard baseline and it is not new machinery — it is the tree you
already understand, averaged.

Note the baseline row. Every other model must be read against it.

### 15.4 Tune the two best, still without the test set

In [ ]:
tuning_grids = {
    "SVM, RBF": (
        make_pipeline(StandardScaler(), SVC(random_state=SEED)),
        {"svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", 0.01, 0.1, 1.0]},
    ),
    "random forest": (
        make_pipeline(StandardScaler(), RandomForestClassifier(random_state=SEED)),
        {"randomforestclassifier__n_estimators": [200],
         "randomforestclassifier__max_depth": [4, 6, 8, None],
         "randomforestclassifier__min_samples_leaf": [1, 3, 5]},
    ),
}

tuned = {}
for name, (estimator, grid) in tuning_grids.items():
    search = GridSearchCV(estimator, grid, cv=final_cv, scoring="accuracy", refit=True)
    search.fit(X_dev, y_dev)
    tuned[name] = search
    print(f"{name}")
    print(f"  combinations : {len(search.cv_results_['params'])}")
    print(f"  best params  : {search.best_params_}")
    print(f"  best CV score: {search.best_score_:.4f}")

In [ ]:
best_name = max(tuned, key=lambda n: tuned[n].best_score_)
best_model = tuned[best_name]

print(f"selected on cross-validated accuracy: {best_name}")
print(f"its cross-validated accuracy        : {best_model.best_score_:.4f}")
print()
print("This is the last decision. Everything from here uses the test set,")
print("and only to measure, never to choose.")

### 15.5 Open the test set, once

In [ ]:
y_test_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)
baseline_test = DummyClassifier(strategy="most_frequent").fit(X_dev, y_dev).score(X_test, y_test)

print(f"model          : {best_name}")
print(f"test accuracy  : {test_accuracy:.4f}")
print(f"baseline       : {baseline_test:.4f}")
print(f"improvement    : {test_accuracy - baseline_test:+.4f}")
print(f"cross-validated estimate was {best_model.best_score_:.4f}, "
      f"difference {test_accuracy - best_model.best_score_:+.4f}")
print()
print(classification_report(y_test, y_test_pred, target_names=["died", "survived"], digits=4))

In [ ]:
final_matrix = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(final_matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["pred died", "pred survived"],
            yticklabels=["actual died", "actual survived"], ax=ax)
ax.set_title(f"{best_name} on the held-out test set")
plt.show()

tn, fp, fn, tp = final_matrix.ravel()
print(f"TN {tn}   FP {fp} (false alarms)   FN {fn} (missed)   TP {tp}")

The cross-validated estimate and the test score are close, which is the
outcome you want: it means the development procedure was not fitting
itself to the development data.

### 15.6 The comparison table, and a decision

In [ ]:
final_rows = []
for name, model in candidates.items():
    cv_row = comparison_table.loc[name]
    if name in tuned:
        fitted, cv_accuracy, note = tuned[name].best_estimator_, tuned[name].best_score_, "tuned"
    else:
        fitted = model.fit(X_dev, y_dev)
        cv_accuracy, note = cv_row["cv_accuracy"], "defaults"
    final_rows.append({
        "model": name,
        "settings": note,
        "cv_accuracy": cv_accuracy,
        "cv_std": cv_row["cv_std"],
        "test_accuracy": fitted.score(X_test, y_test),
    })

final_table = pd.DataFrame(final_rows).set_index("model").sort_values("cv_accuracy", ascending=False)
print(final_table.round(4).to_string())

**A note on the last column.** It is printed here so the lesson is
complete, and computing it for every candidate is exactly the practice
§10.4 warned against. Having now seen six test scores, no further model
choice in this notebook can be reported as unbiased. In real work you
compute that column for the selected model alone.

### The decision

Look at that table carefully before reading on, because it contains a
trap and the trap is the most useful thing in this notebook.

**Ranked by cross-validated accuracy**, the tuned random forest is first
at 0.820 and logistic regression is last of the fitted models at 0.795.
**Ranked by test accuracy**, logistic regression is first at 0.849 and
the random forest is fourth at 0.816. The two orderings are almost
reversed.

Nothing has gone wrong. The test set is 179 rows, so one standard error
on an accuracy near 0.82 is about 0.029 — and every fitted model in that
table sits inside a band roughly that wide. **The differences between
them are not resolved by this much data**, in either column, and the
apparent reversal is what noise of that size looks like.

If you had chosen on the test column you would now be reporting logistic
regression at 0.849, and that number would be the maximum of six noisy
draws rather than an estimate of anything (§10.4).

So: **deploy the tuned random forest**, which is what the development
procedure selected before the test set was opened, and report 0.816 as
its performance. The reasoning, in the order it should be given:

1. **Everything beats the baseline by a wide margin.** The majority-class
   rule scores 0.615; every fitted model is 18 to 23 points above it, so
   there is real signal and the exercise was worth doing.

2. **The candidates are not separated by the evidence.** Their
   cross-validated spreads overlap heavily — the forest's own
   fold-to-fold standard deviation is 0.015, and the gap between it and
   the second model is 0.003. Calling the top row the best model is
   over-reading the data.

3. **So break the tie on something other than the score.** The forest was
   selected by the procedure, and it also happens to be defensible on
   other grounds: it is the most stable across folds (the smallest
   `cv_std` of any fitted model), it needs no scaling to work, and it
   reports feature importances. Had the requirement been a model a
   non-specialist could read end to end, the depth-4 tree is one point
   behind and fully inspectable, and that would be a reasonable trade.

4. **Recall on survivors matters more than accuracy here.** The classes
   are imbalanced (§9.4). State which error is worse for the application
   and choose an operating point accordingly, rather than accepting the
   default threshold of 0.5.

5. **What would change this.** More data, particularly on the passengers
   with missing ages; a feature capturing family groups travelling
   together; and an explicit cost for each error type, which would move
   the threshold and could change the ranking outright.

In [ ]:
forest = tuned["random forest"].best_estimator_.named_steps["randomforestclassifier"]
importances = pd.Series(forest.feature_importances_, index=final_feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
importances.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel("mean decrease in impurity")
ax.set_title("What the forest used")
plt.show()

print(importances.round(4).to_string())

`Sex` and `Fare` dominate, with `Age` and `Pclass` behind them. That is
consistent with everything Lesson 2 found by inspection — and note that
`Fare` ranks highly for the reason Practice 2 §12.1 dissected: it is
partly acting as a proxy for `Pclass`.

Treat impurity-based importances as a description of what this model
used, not as a measure of causal importance. They are biased toward
high-cardinality features, and two correlated features will split the
credit between them arbitrarily.

## 16. Discussion

**16.1 — Your model scores 0.94 on the test set and 0.99 on the training
set. Is it overfitting?**

The gap is evidence of it, and the gap alone does not settle it (§7.3).
Compare both against a baseline and against the ceiling: if 0.94 is near
the achievable maximum, the model is fine and merely flexible. If a
simpler model also reaches 0.94, prefer the simpler one.

**16.2 — Why is the test score of a model whose hyperparameters were
chosen on the test set biased upward, even though every line of code was
correct?**

Because the reported number is the maximum of many noisy measurements,
and the maximum of noisy measurements exceeds the expected value of any
one of them (§10.4). Nothing in the code is wrong; the procedure
contains the error.

**16.3 — RMSE is 102 and MAE is 76 on the bicycle data. What does the
ratio tell you?**

That errors are unevenly sized: a minority of hours are badly wrong and
pull the squared term up (§4.2). If the two were nearly equal the errors
would be uniform. This directs you to look at *which* hours fail, which
the residual plot answers.

**16.4 — A classifier reports 99 % accuracy on a dataset where 1 % of
cases are positive. What do you ask for?**

The confusion matrix, and precision and recall on the positive class
(§9.4). Always predicting "negative" scores 99 % with zero recall, so
accuracy alone cannot distinguish a perfect model from a useless one.

**16.5 — A colleague scales the whole dataset, then cross-validates, and
reports a higher score than you. What happened, and how large is the
effect?**

Each fold's validation rows contributed to the scaling parameters applied
to its training rows (§6.3). The size depends on how much the step
learns: for a scaler on well-behaved data it can be invisible, and for a
feature selector on many columns it produced 0.77 accuracy on pure noise.

**16.6 — Your SVM with `C=50, gamma=5` beats every other model on
training data. What do you check before believing it?**

Its cross-validated score, and the gap between that and the training
score (§12.3). On four of the six kernel datasets the wiggly model's
advantage disappeared or reversed. On two it survived — high `gamma` is a
claim that the boundary is local, and that claim is sometimes true.

**16.7 — Your clustering has a silhouette of 0.71. Is that good?**

It is a number, not a verdict. Silhouette rewards compact, well-separated
spherical clusters, so a high value on elongated or overlapping data may
mean the algorithm imposed a shape rather than found one (§14.5). If the
model has a likelihood, check BIC as well: on the three-component data
above, silhouette said 2 and BIC said 3, and BIC was right.

**16.8 — You cluster customers, find four groups, and are asked whether
they are real. What can you answer?**

That the algorithm partitioned the data into four groups, that internal
measures did or did not favour four over three and five, and that
stability under reseeding and resampling does or does not hold (§13.1,
§13.4). You cannot answer that the groups are real: there is no ground
truth, and k-means returns `k` clusters for any `k` you pass.

**16.9 — Two models score 0.834 and 0.829 in cross-validation. Which is
better?**

On this evidence, neither. Report the spread across folds: if the
standard deviation is 0.02, a difference of 0.005 is noise (§6.2). Break
the tie on interpretability, cost, latency or maintenance, and say that
is what you did.

## 17. Summary

- Every estimator is `fit` / `predict` / `transform`, and anything
  learned carries a trailing underscore (§2).
- A model can be graded against known parameters when you generated the
  data, and against held-out data when you did not (§3, §5).
- RMSE is in the units of the target, MAE ignores the size of large
  errors, R² is unitless; and a residual plot says things no metric does
  (§4).
- The test set is a measurement instrument with one use (§5, §10.4).
- Cross-validation reports a distribution, not a number, and every step
  that learns from data belongs inside the `Pipeline` (§6).
- Training error always falls with flexibility; validation error does
  not, and the gap between them is the diagnosis (§7, §12.3).
- Trees are readable and scale-invariant, and grow to meaningless purity
  unless constrained (§8).
- Build precision and recall from the confusion matrix, and state the
  majority-class baseline before quoting any accuracy (§9).
- An SVM maximises the margin; `C` prices violations, `gamma` sets the
  reach of one point, and a model fitted on scaled data reports its
  parameters in scaled space (§11, §12).
- Clustering has no ground truth. Separate internal from external
  validation, match the criterion to the model, and remember that
  k-means will return `k` clusters whatever you ask for (§13, §14).
- A result is a decision with reasons attached, not the top row of a
  leaderboard (§15.6).

### Further reading

- scikit-learn user guide — <https://scikit-learn.org/stable/user_guide.html>
- Common pitfalls and recommended practices —
  <https://scikit-learn.org/stable/common_pitfalls.html>
- Cross-validation: evaluating estimator performance —
  <https://scikit-learn.org/stable/modules/cross_validation.html>
- Hastie, Tibshirani and Friedman, *The Elements of Statistical
  Learning*, 2nd edition — <https://hastie.su.domains/ElemStatLearn/>.
  Section 7.10.2 is the feature-selection leak of §6.3.
- James, Witten, Hastie and Tibshirani, *An Introduction to Statistical
  Learning* — <https://www.statlearning.com/>

In [ ]:
print(f"notebook executed end to end in {time.time() - NOTEBOOK_START:.1f} seconds")